# Spiking-network benchmarks (GPU)

Benchmarks a **recurrent spiking neural network** with excitatory and inhibitory
neurons on a noisy burst-detection task, sweeping five things: **network size,
E/I ratio, neuron type, learning method and sparsity**. Learning rates are tuned
beforehand by random search on a validation set. Every figure is drawn inline
and followed by the table behind it; nothing is downloaded.

**How to use**
1. *Runtime > Change runtime type > GPU* (a T4 is enough).
2. Leave **`QUICK_RUN = True`** for a first pass (a few minutes) to check that
   everything works, then set it to `False` and re-run for the real result.
3. *Runtime > Run all*, or run the cells top to bottom. Each sweep has its own
   cell, so you can re-run one on its own.

**About GPU speed.** All tensors, weights, traces and learning-rule updates live on
the GPU, and the input current and the readout are single large matrix products
over all time steps. The recurrent update itself is sequential in time, so for
small networks it is limited by kernel-launch overhead rather than arithmetic;
the GPU pays off most for the large networks and densities in the sweeps. Set
`DEVICE_OVERRIDE = "cpu"` to compare.

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Use Runtime > Change runtime type > GPU, or continue on CPU (slower).")

In [ ]:
#@title Run settings { display-mode: "form" }
QUICK_RUN = True            #@param {type:"boolean"}
DEVICE_OVERRIDE = "auto"    #@param ["auto", "cuda", "cpu"]
SEEDS_OVERRIDE = 0          #@param {type:"integer"}
SAVE_TO_DRIVE = False       #@param {type:"boolean"}
# QUICK_RUN: tiny budget, a few minutes: checks the pipeline, numbers are not meaningful.
# SEEDS_OVERRIDE: 0 keeps the budget default (3 seeds full, 1 quick).
# SAVE_TO_DRIVE: keep results.json / hparams.json (and PNG + CSV copies) in Google Drive so a
#                disconnected session can resume. Off: nothing is written outside this VM.

## Library code
The neuron models and the network are written to files so they can be imported. They are embedded verbatim from `neurons.py` and `rsnn.py`.

In [ ]:
%%writefile neurons.py
"""
Interchangeable spiking-neuron populations (PyTorch).

Every class is a *population*: one object holds the state of ``n`` neurons as
tensors and advances all of them in a single vectorised step. There are no
per-neuron Python objects, so cost scales with tensor kernels, not interpreter
overhead, and the same code runs on CPU or GPU.

Common interface (what a network class should rely on)
------------------------------------------------------
    neuron = make_neuron("lif", n=128, dt=1.0, ...)   # or LIF(128, ...)
    neuron.reset_state()                    # forget state (new sequence/batch)
    spikes = neuron(current)                # one step; current: (n,) or (B, n)
    spikes = neuron.sequence(currents)      # (T, B, n) or (T, n) -> same shape
    neuron.detach_state()                   # cut the graph (truncated BPTT)
    neuron.membrane                         # last membrane variable (or None)
    neuron.n                                # population size

Spikes are returned as floats (0./1.) with the dtype/device of the input.

Units
-----
Time is in milliseconds (``dt``, ``tau_*``, ``refractory``); rates in Hz.
IF / LIF / ALIF voltages are *relative to rest*: rest = 0, the default
threshold is 1, and the input is a per-step charge (``v <- beta*v + R*I``), the
same convention as snnTorch, so these three are drop-in interchangeable.
``Izhikevich`` uses the model's native mV scale and current units.

Models
------
    IFNeuron            non-leaky integrate-and-fire
    LIF                 leaky integrate-and-fire (optionally learnable tau)
    AdaptiveLIF         LIF with an adaptive threshold (spike-frequency
                        adaptation; long memory for recurrent nets)
    Izhikevich          two-variable model; presets for RS/IB/CH/FS/LTS/TC/RZ
    StochasticLIF       LIF with escape noise (probabilistic spiking)
    PoissonNeuron       input is a rate in Hz; spikes are Bernoulli draws

Any model also accepts ``noise_std`` (diffusive Gaussian input noise,
Euler-Maruyama scaled by sqrt(dt)), making every deterministic model stochastic.

Training
--------
Deterministic spikes use a fast-sigmoid surrogate gradient in the backward
pass. Stochastic spikes use a straight-through estimator through the spike
probability. The reset is detached from the graph (standard practice).

Efficiency notes
----------------
* Scalar parameters stay Python floats (no tensor allocation per step);
  per-neuron parameters are buffers and broadcast.
* With autograd disabled (``torch.no_grad()`` / ``inference_mode()``) steps run
  *in place* on the state tensors; with autograd enabled they run out of place
  so BPTT stays correct. Call ``reset_state()`` when switching between modes.
* ``sequence`` writes into one preallocated output tensor.
"""

from __future__ import annotations

import math
from abc import ABC, abstractmethod
from typing import Dict, Optional, Tuple, Union

import torch
from torch import Tensor, nn

__all__ = [
    "SpikingNeuron",
    "ThresholdNeuron",
    "IFNeuron",
    "LIF",
    "AdaptiveLIF",
    "Izhikevich",
    "StochasticLIF",
    "PoissonNeuron",
    "NEURON_TYPES",
    "make_neuron",
]

Param = Union[float, Tensor]
State = Dict[str, Tensor]


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
class _FastSigmoidSpike(torch.autograd.Function):
    """Heaviside forward, fast-sigmoid derivative 1/(slope*|x|+1)^2 backward."""

    @staticmethod
    def forward(ctx, x: Tensor, slope: float) -> Tensor:
        ctx.save_for_backward(x)
        ctx.slope = slope
        return (x >= 0).to(x.dtype)

    @staticmethod
    def backward(ctx, grad_out: Tensor):
        (x,) = ctx.saved_tensors
        denom = ctx.slope * x.abs() + 1.0
        return grad_out / (denom * denom), None


def spike_fn(x: Tensor, slope: float) -> Tensor:
    """Spike where ``x >= 0``; surrogate gradient only when one is needed."""
    if x.requires_grad and torch.is_grad_enabled():
        return _FastSigmoidSpike.apply(x, slope)
    return (x >= 0).to(x.dtype)


def _register(module: nn.Module, name: str, value: Param) -> None:
    """Tensors become buffers (move with .to(), broadcast per neuron);
    scalars stay Python floats (fastest path, no allocation per step)."""
    if isinstance(value, Tensor):
        module.register_buffer(name, value.detach().clone().to(torch.get_default_dtype()))
    else:
        setattr(module, name, float(value))


def _is_one(x: Param) -> bool:
    return isinstance(x, float) and x == 1.0


def _mul(x: Tensor, c: Param) -> Tensor:
    return x if _is_one(c) else x * c


def _axpy_(dst: Tensor, src: Tensor, coef: Param) -> Tensor:
    """dst += coef * src, in place, for float or per-neuron-tensor coef.

    Deliberately a separate multiply then add (no ``alpha=`` / ``addcmul_``):
    those can use fused multiply-add, which rounds differently from the
    out-of-place ``dst + coef * src``. Threshold dynamics amplify that ~1 ulp
    difference into different spike times, so both paths must match bitwise.
    """
    return dst.add_(src if _is_one(coef) else src * coef)


def _fill(v: Tensor, mask: Tensor, value: Param, inplace: bool) -> Tensor:
    """Set v[mask] = value (float or per-neuron tensor)."""
    if inplace:
        if isinstance(value, Tensor):
            return v.copy_(torch.where(mask, value, v))
        return v.masked_fill_(mask, value)
    return torch.where(mask, value, v)


# --------------------------------------------------------------------------- #
# Base class: the polymorphic interface
# --------------------------------------------------------------------------- #
class SpikingNeuron(nn.Module, ABC):
    """Abstract population of ``n`` spiking neurons.

    Subclasses implement ``_init_state`` and ``_step``; everything a network
    needs (state handling, noise, sequences, truncated-BPTT support) lives here.
    """

    def __init__(
        self,
        n: int,
        dt: float = 1.0,
        noise_std: float = 0.0,
        surrogate_slope: float = 25.0,
    ):
        super().__init__()
        self.n = int(n)
        self.dt = float(dt)
        self.noise_std = float(noise_std)
        self.surrogate_slope = float(surrogate_slope)
        self._state: Optional[State] = None
        self._keep_pre = False  # record pre-reset voltage (used by sequence())

    # ---- to implement --------------------------------------------------- #
    @abstractmethod
    def _init_state(self, like: Tensor) -> State:
        """Fresh state matching ``like`` (shape/dtype/device). Must contain 'v'."""

    @abstractmethod
    def _step(self, current: Tensor, state: State) -> Tuple[Tensor, State]:
        """Advance one time step. Returns (spikes, new_state)."""

    # ---- shared behaviour ----------------------------------------------- #
    def reset_state(self) -> None:
        self._state = None

    def detach_state(self) -> None:
        if self._state is not None:
            self._state = {k: t.detach() for k, t in self._state.items()}

    @property
    def membrane(self) -> Optional[Tensor]:
        if self._state is None:
            return None
        return self._state.get("v_pre", self._state["v"])

    @property
    def record_pre_reset(self) -> bool:
        """When True, ``membrane`` returns the pre-reset voltage of the last step."""
        return self._keep_pre

    @record_pre_reset.setter
    def record_pre_reset(self, flag: bool) -> None:
        self._keep_pre = bool(flag)

    def pseudo_derivative(self, v_pre: Tensor) -> Tensor:
        """d(spike)/d(v) used as the post-synaptic factor of local learning
        rules (e-prop / three-factor). ``v_pre`` is the pre-reset membrane."""
        raise NotImplementedError(f"{type(self).__name__} has no pseudo-derivative")

    def forward(self, current: Tensor) -> Tensor:
        st = self._state
        if st is not None:
            v = st["v"]
            if v.shape != current.shape or v.dtype != current.dtype or v.device != current.device:
                st = None
        if st is None:
            st = self._init_state(current)
        if self.noise_std > 0.0:
            current = current + torch.randn_like(current).mul_(self.noise_std * math.sqrt(self.dt))
        spikes, self._state = self._step(current, st)
        return spikes

    def sequence(self, currents: Tensor, record_membrane: bool = False):
        """Run T steps. ``currents``: (T, ..., n). Returns spikes (and the
        pre-reset membrane trace if ``record_membrane``)."""
        if currents.shape[-1] != self.n:
            raise ValueError(f"expected last dim {self.n}, got {tuple(currents.shape)}")
        spikes = torch.empty_like(currents)
        trace = torch.empty_like(currents) if record_membrane else None
        self._keep_pre = record_membrane
        try:
            for t in range(currents.shape[0]):
                spikes[t] = self(currents[t])
                if trace is not None:
                    trace[t] = self.membrane
        finally:
            self._keep_pre = False
        return (spikes, trace) if record_membrane else spikes

    def extra_repr(self) -> str:
        return f"n={self.n}, dt={self.dt}"


# --------------------------------------------------------------------------- #
# Threshold-and-reset family: IF, LIF, ALIF, StochasticLIF
# --------------------------------------------------------------------------- #
class ThresholdNeuron(SpikingNeuron):
    """Shared machinery: integrate -> (refractory) -> fire -> reset.

    Subclasses customise ``_beta`` (leak) and/or ``_fire`` (spike generation).
    """

    def __init__(
        self,
        n: int,
        v_threshold: Param = 1.0,
        v_reset: Param = 0.0,
        reset_mode: str = "hard",
        refractory: float = 0.0,
        resistance: Param = 1.0,
        **kw,
    ):
        super().__init__(n, **kw)
        if reset_mode not in ("hard", "soft"):
            raise ValueError("reset_mode must be 'hard' or 'soft'")
        _register(self, "v_threshold", v_threshold)
        _register(self, "v_reset", v_reset)
        _register(self, "resistance", resistance)
        self.reset_mode = reset_mode
        self.refractory_steps = int(round(refractory / self.dt))

    # ---- hooks ----------------------------------------------------------- #
    def _beta(self) -> Param:
        return 1.0  # no leak

    def _fire(self, v: Tensor, st: State) -> Tensor:
        return spike_fn(v - self.v_threshold, self.surrogate_slope)

    def pseudo_derivative(self, v_pre: Tensor) -> Tensor:
        d = self.surrogate_slope * (v_pre - self.v_threshold).abs() + 1.0
        return 1.0 / (d * d)

    # ---- core ------------------------------------------------------------ #
    def _init_state(self, like: Tensor) -> State:
        st = {"v": torch.zeros_like(like)}
        if self.refractory_steps:
            st["ref"] = torch.zeros_like(like)
        return st

    def _step(self, current: Tensor, st: State) -> Tuple[Tensor, State]:
        inplace = not torch.is_grad_enabled()
        v = st["v"]
        nref = self.refractory_steps

        # integrate: v <- beta * v + R * I
        beta = self._beta()
        if inplace:
            if not _is_one(beta):
                v.mul_(beta)
            _axpy_(v, current, self.resistance)
        else:
            v = _mul(v, beta) + _mul(current, self.resistance)

        # refractory: clamp membrane to reset value while the counter is active
        if nref:
            ref = st["ref"]
            in_ref = ref > 0
            v = _fill(v, in_ref, self.v_reset, inplace)

        if self._keep_pre:
            st["v_pre"] = v.clone() if inplace else v
        else:
            st.pop("v_pre", None)

        # fire
        s = self._fire(v, st)
        if nref:
            s = s * (~in_ref)
        m = s.detach() > 0.5  # reset is detached from the graph

        # reset
        if self.reset_mode == "hard":
            v = _fill(v, m, self.v_reset, inplace)
        elif inplace:
            v.sub_(m.to(v.dtype) * self.v_threshold)
        else:
            v = v - m.to(v.dtype) * self.v_threshold

        if nref:
            ref.sub_(1.0).clamp_(min=0.0)
            ref.masked_fill_(m, float(nref))

        st["v"] = v
        return s, st


class IFNeuron(ThresholdNeuron):
    """Non-leaky integrate-and-fire: v <- v + R*I; spike at threshold."""


class LIF(ThresholdNeuron):
    """Leaky integrate-and-fire: v <- exp(-dt/tau_m) * v + R*I.

    ``tau_m`` may be a float or a per-neuron tensor. With ``learnable_tau`` the
    time constants become trainable (stored as log tau, per neuron).
    """

    def __init__(self, n: int, tau_m: Param = 20.0, learnable_tau: bool = False, **kw):
        super().__init__(n, **kw)
        self.learnable_tau = learnable_tau
        if learnable_tau:
            tau = torch.as_tensor(tau_m, dtype=torch.get_default_dtype()).expand(self.n)
            self.log_tau = nn.Parameter(tau.clone().log())
        else:
            if isinstance(tau_m, Tensor):
                beta: Param = torch.exp(-self.dt / tau_m)
            else:
                beta = math.exp(-self.dt / float(tau_m))
            _register(self, "_beta_const", beta)

    def _beta(self) -> Param:
        if self.learnable_tau:
            return torch.exp(-self.dt / torch.exp(self.log_tau))
        return self._beta_const


class AdaptiveLIF(LIF):
    """LIF with adaptive threshold: theta_eff = theta + strength * a,
    a <- exp(-dt/tau_adapt) * a + spike. Slow adaptation gives recurrent nets
    long memory and richer oscillatory dynamics."""

    def __init__(self, n: int, tau_adapt: float = 200.0, adapt_strength: Param = 0.2, **kw):
        super().__init__(n, **kw)
        self._rho = math.exp(-self.dt / float(tau_adapt))
        _register(self, "adapt_strength", adapt_strength)

    def _init_state(self, like: Tensor) -> State:
        st = super()._init_state(like)
        st["a"] = torch.zeros_like(like)
        return st

    def _fire(self, v: Tensor, st: State) -> Tensor:
        theta = self.v_threshold + self.adapt_strength * st["a"]
        s = spike_fn(v - theta, self.surrogate_slope)
        st["a"] = st["a"] * self._rho + s
        return s


class StochasticLIF(LIF):
    """LIF with escape noise: spikes are Bernoulli draws from a probability
    that increases with the membrane potential.

    escape='sigmoid': p = sigmoid((v - theta) / delta)
    escape='exp'    : rate = rho0 * exp((v - theta) / delta) Hz,
                      p = 1 - exp(-rate * dt)
    ``delta`` (same units as v) sets the softness; delta -> 0 recovers LIF.
    Gradients use a straight-through estimator through p.
    """

    def __init__(
        self,
        n: int,
        delta: Param = 0.1,
        escape: str = "sigmoid",
        rho0: float = 10.0,
        **kw,
    ):
        super().__init__(n, **kw)
        if escape not in ("sigmoid", "exp"):
            raise ValueError("escape must be 'sigmoid' or 'exp'")
        self.escape = escape
        self.rho0 = float(rho0)
        _register(self, "delta", delta)

    def _fire(self, v: Tensor, st: State) -> Tensor:
        x = (v - self.v_threshold) / self.delta
        if self.escape == "sigmoid":
            p = torch.sigmoid(x)
        else:
            rate = self.rho0 * torch.exp(x.clamp(max=30.0))
            p = -torch.expm1(rate * (-self.dt * 1e-3))
        bern = (torch.rand_like(p) < p).to(p.dtype)
        if p.requires_grad and torch.is_grad_enabled():
            return bern + p - p.detach()  # value = bern, gradient = dp
        return bern

    def pseudo_derivative(self, v_pre: Tensor) -> Tensor:
        """dp/dv of the escape function (the true derivative of the spike
        probability, which is what the straight-through gradient uses)."""
        x = (v_pre - self.v_threshold) / self.delta
        if self.escape == "sigmoid":
            p = torch.sigmoid(x)
            return p * (1.0 - p) / self.delta
        rate_dt = self.rho0 * torch.exp(x.clamp(max=30.0)) * (self.dt * 1e-3)
        return torch.exp(-rate_dt) * rate_dt / self.delta


# --------------------------------------------------------------------------- #
# Izhikevich
# --------------------------------------------------------------------------- #
class Izhikevich(SpikingNeuron):
    """Izhikevich (2003) model, native mV / ms units:

        dv/dt = 0.04 v^2 + 5 v + 140 - u + I
        du/dt = a (b v - u)
        if v >= v_peak:  v <- c,  u <- u + d

    ``a, b, c, d`` may be floats or per-neuron tensors (heterogeneous
    populations). ``substeps`` splits each ``dt`` into smaller Euler steps for
    v (Izhikevich's original code uses 2 x 0.5 ms for dt = 1 ms).
    """

    PRESETS = {
        "RS": (0.02, 0.20, -65.0, 8.00),    # regular spiking
        "IB": (0.02, 0.20, -55.0, 4.00),    # intrinsically bursting
        "CH": (0.02, 0.20, -50.0, 2.00),    # chattering
        "FS": (0.10, 0.20, -65.0, 2.00),    # fast spiking
        "LTS": (0.02, 0.25, -65.0, 2.00),   # low-threshold spiking
        "TC": (0.02, 0.25, -65.0, 0.05),    # thalamo-cortical
        "RZ": (0.10, 0.26, -65.0, 2.00),    # resonator
    }
    V_CLIP = 1.0e3  # guards against float overflow under extreme drive

    def __init__(
        self,
        n: int,
        a: Param = 0.02,
        b: Param = 0.20,
        c: Param = -65.0,
        d: Param = 8.0,
        v_peak: float = 30.0,
        substeps: int = 2,
        surrogate_slope: float = 0.5,  # voltage scale is mV, not ~1
        **kw,
    ):
        super().__init__(n, surrogate_slope=surrogate_slope, **kw)
        for name, val in (("a", a), ("b", b), ("c", c), ("d", d)):
            _register(self, name, val)
        self.v_peak = float(v_peak)
        self.substeps = int(substeps)

    @classmethod
    def from_preset(cls, name: str, n: int, **kw) -> "Izhikevich":
        a, b, c, d = cls.PRESETS[name]
        return cls(n, a=a, b=b, c=c, d=d, **kw)

    @classmethod
    def cortical_population(
        cls, n_exc: int, n_inh: int, seed: Optional[int] = None, **kw
    ) -> "Izhikevich":
        """Heterogeneous E/I population from Izhikevich (2003); excitatory
        neurons come first."""
        g = torch.Generator().manual_seed(seed) if seed is not None else None
        re = torch.rand(n_exc, generator=g)
        ri = torch.rand(n_inh, generator=g)
        a = torch.cat([torch.full((n_exc,), 0.02), 0.02 + 0.08 * ri])
        b = torch.cat([torch.full((n_exc,), 0.20), 0.25 - 0.05 * ri])
        c = torch.cat([-65.0 + 15.0 * re ** 2, torch.full((n_inh,), -65.0)])
        d = torch.cat([8.0 - 6.0 * re ** 2, torch.full((n_inh,), 2.0)])
        return cls(n_exc + n_inh, a=a, b=b, c=c, d=d, **kw)

    def _init_state(self, like: Tensor) -> State:
        v = torch.zeros_like(like).add_(self.c)
        u = v * self.b
        return {"v": v, "u": u}

    def pseudo_derivative(self, v_pre: Tensor) -> Tensor:
        d = self.surrogate_slope * (v_pre - self.v_peak).abs() + 1.0
        return 1.0 / (d * d)

    def _step(self, current: Tensor, st: State) -> Tuple[Tensor, State]:
        inplace = not torch.is_grad_enabled()
        v, u = st["v"], st["u"]
        h = self.dt / self.substeps

        if inplace:
            for _ in range(self.substeps):
                dv = torch.mul(v, 0.04).add_(5.0).mul_(v).add_(140.0).sub_(u).add_(current)
                # same op order as the out-of-place path: v + (h * dv)
                v.add_(dv.mul_(h)).clamp_(max=self.V_CLIP)
            du = torch.mul(v, self.b).sub_(u)
            u.add_(du.mul_(self.a * self.dt))  # u + (a*dt) * du, unfused
        else:
            for _ in range(self.substeps):
                v = (v + h * (v * (0.04 * v + 5.0) + 140.0 - u + current)).clamp(max=self.V_CLIP)
            u = u + (self.a * self.dt) * (v * self.b - u)

        if self._keep_pre:
            st["v_pre"] = (v.clone() if inplace else v).clamp(max=self.v_peak)
        else:
            st.pop("v_pre", None)

        s = spike_fn(v - self.v_peak, self.surrogate_slope)
        m = s.detach() > 0.5

        if inplace:
            _fill(v, m, self.c, True)
            u.add_(m.to(u.dtype) * self.d)
        else:
            v = torch.where(m, self.c, v)
            u = u + m.to(u.dtype) * self.d

        st["v"], st["u"] = v, u
        return s, st


# --------------------------------------------------------------------------- #
# Poisson
# --------------------------------------------------------------------------- #
class PoissonNeuron(SpikingNeuron):
    """Stochastic rate-driven neurons: the input is the firing rate in Hz and
    each step spikes with probability 1 - exp(-rate * dt). Stateless apart from
    ``membrane`` (the last spike probability). Straight-through gradient."""

    def _init_state(self, like: Tensor) -> State:
        return {"v": torch.zeros_like(like)}

    def _step(self, rate: Tensor, st: State) -> Tuple[Tensor, State]:
        p = -torch.expm1(rate.clamp(min=0.0) * (-self.dt * 1e-3))
        bern = (torch.rand_like(p) < p).to(p.dtype)
        s = bern + p - p.detach() if (p.requires_grad and torch.is_grad_enabled()) else bern
        st["v"] = p
        return s, st


# --------------------------------------------------------------------------- #
# Factory (lets a network class pick the neuron model by name)
# --------------------------------------------------------------------------- #
NEURON_TYPES = {
    "if": IFNeuron,
    "lif": LIF,
    "alif": AdaptiveLIF,
    "izhikevich": Izhikevich,
    "stochastic_lif": StochasticLIF,
    "poisson": PoissonNeuron,
}


def make_neuron(kind: str, n: int, **kwargs) -> SpikingNeuron:
    try:
        cls = NEURON_TYPES[kind.lower()]
    except KeyError:
        raise ValueError(f"unknown neuron type {kind!r}; choose from {sorted(NEURON_TYPES)}")
    return cls(n, **kwargs)


# --------------------------------------------------------------------------- #
# Self-test:  python neurons.py
# --------------------------------------------------------------------------- #
def _selftest() -> None:
    torch.manual_seed(0)
    T, B, N = 200, 4, 16

    def drive(kind: str) -> Tensor:
        if kind == "izhikevich":
            return 10.0 + 2.0 * torch.randn(T, B, N)
        if kind == "poisson":
            return 40.0 * torch.rand(T, B, N)  # Hz
        return 0.12 + 0.05 * torch.randn(T, B, N)

    kwargs = {
        "if": dict(refractory=2.0),
        "lif": dict(tau_m=20.0, refractory=2.0),
        "alif": dict(tau_m=20.0),
        "izhikevich": dict(),
        "stochastic_lif": dict(tau_m=20.0, delta=0.1),
        "poisson": dict(),
    }

    for kind, kw in kwargs.items():
        neuron = make_neuron(kind, N, **kw)
        x = drive(kind)

        # 1. shapes + substitutability through the common interface
        with torch.no_grad():
            neuron.reset_state()
            torch.manual_seed(1)
            fast = neuron.sequence(x)
        assert fast.shape == x.shape and set(fast.unique().tolist()) <= {0.0, 1.0}, kind

        # 2. in-place (no_grad) and out-of-place (autograd) paths agree exactly.
        #    Several seeds: a ~1 ulp rounding difference only flips a spike in
        #    some runs, so a single seed can hide a mismatch.
        for seed in range(8):
            torch.manual_seed(100 + seed)
            xs = drive(kind)
            with torch.no_grad():
                neuron.reset_state()
                torch.manual_seed(1)
                a_ = neuron.sequence(xs)
            neuron.reset_state()
            torch.manual_seed(1)
            b_ = neuron.sequence(xs)
            assert torch.equal(a_, b_), (
                f"{kind}: inplace/out-of-place mismatch (seed {seed}, "
                f"{int((a_ != b_).sum())} differing entries)"
            )

        # 3. gradients flow back to the input
        neuron.reset_state()
        xr = x.clone().requires_grad_(True)
        neuron.sequence(xr).sum().backward()
        assert xr.grad is not None and torch.isfinite(xr.grad).all(), kind

        # 4. membrane recording + state handling
        neuron.reset_state()
        with torch.no_grad():
            _, trace = neuron.sequence(x, record_membrane=True)
        assert trace.shape == x.shape
        neuron.detach_state()

        print(f"{kind:15s} ok   mean rate = {fast.mean().item() * 1000 / neuron.dt:7.1f} Hz")

    # learnable time constants receive gradients
    lif = LIF(N, tau_m=20.0, learnable_tau=True)
    lif.sequence(drive("lif")).sum().backward()
    assert lif.log_tau.grad is not None and torch.isfinite(lif.log_tau.grad).all()

    # heterogeneous population + presets construct and run
    pop = Izhikevich.cortical_population(12, 4, seed=0)
    with torch.no_grad():
        assert pop.sequence(10.0 + torch.randn(50, 16)).shape == (50, 16)
    Izhikevich.from_preset("FS", 8)

    print("all self-tests passed")


if __name__ == "__main__":
    _selftest()

In [ ]:
%%writefile rsnn.py
"""
Recurrent spiking neural network (RSNN) with excitatory/inhibitory populations
and a family of interchangeable learning rules.

Network
-------
``RSNN`` builds a sparse recurrent network of ``n_rec`` neurons, a fraction
``exc_fraction`` of them excitatory (indices ``[0, n_exc)``) and the rest
inhibitory. Neuron models come from ``neurons.py`` and can be swapped by name,
class, factory or pre-built instance; excitatory and inhibitory neurons may use
different models. Size, sparsity (a single density or a separate one for each of
EE/EI/IE/II), Dale's law, spectral radius, synaptic filtering and readout are
all configurable.

    net = RSNN(n_in=20, n_rec=200, n_out=3, exc_fraction=0.8, connectivity=0.1,
               neuron="alif", inh_neuron="lif")
    y = net(x)                                  # x: (T, B, n_in) -> (T, B, n_out)
    y, rec = net(x, record_membrane=True)       # rec: spikes, membrane, ...

Learning rules (all operate on the same network)
------------------------------------------------
    SurrogateBPTT      backpropagation through time with surrogate gradients
                       (supervised; accepts a fixed *or learnable* loss module)
    STDP               unsupervised pair-based STDP (traces, soft/hard bounds)
    RSTDP              reward-modulated STDP: STDP eligibility x (reward - baseline)
    ThreeFactor        eligibility x post-synaptic pseudo-derivative x learning
                       signal; with the default settings this *is* e-prop
                       (random or symmetric feedback), and any externally
                       supplied modulator turns it into a generic 3-factor rule
    InhibitorySTDP     Vogels et al. 2011 rule for I->E synapses (keeps
                       excitatory neurons near a target rate)
    HomeostaticScaling multiplicative synaptic scaling toward a target rate

Local rules are applied to a *recording* of a chunk (``record=True``), and
chunks can be as short as you like, so updates can be as online as you need.
Rules can be combined, e.g. ThreeFactor for the task plus InhibitorySTDP and
HomeostaticScaling to keep the network in a stable, selective active regime.

Conventions
-----------
Weight matrices are (post, pre). Recurrent spikes arrive with a one-step delay.
With ``dale=True`` the recurrent parameter stores non-negative *magnitudes*;
the effective weight is ``sign(pre) * relu(w) * mask`` so excitatory neurons
never become inhibitory. Fixed sparsity masks are never modified by learning.
Time is in ms (see ``neurons.py``).
"""

from __future__ import annotations

import math
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Callable, Dict, Optional, Sequence, Tuple, Union

import torch
import torch.nn.functional as F
from torch import Tensor, nn

from neurons import SpikingNeuron, make_neuron

__all__ = [
    "RSNN",
    "Recording",
    "Learner",
    "SurrogateBPTT",
    "STDP",
    "RSTDP",
    "ThreeFactor",
    "InhibitorySTDP",
    "HomeostaticScaling",
    "mse_error",
    "softmax_error",
]

NeuronSpec = Union[str, type, Callable[..., SpikingNeuron], SpikingNeuron]


# --------------------------------------------------------------------------- #
# Time-filter helpers (loops over T on small (B, N) tensors; the heavy
# synapse-sized work is always a single matmul over all T*B samples)
# --------------------------------------------------------------------------- #
def _causal_filter(z: Tensor, decay: float) -> Tensor:
    """out[t] = decay * out[t-1] + z[t]."""
    if decay == 0.0:
        return z
    out = torch.empty_like(z)
    acc = torch.zeros_like(z[0])
    for t in range(z.shape[0]):
        acc = acc * decay + z[t]
        out[t] = acc
    return out


def _anticausal_filter(z: Tensor, decay: float) -> Tensor:
    """out[t] = z[t] + decay * out[t+1] (sum over the future)."""
    if decay == 0.0:
        return z
    out = torch.empty_like(z)
    acc = torch.zeros_like(z[-1])
    for t in range(z.shape[0] - 1, -1, -1):
        acc = acc * decay + z[t]
        out[t] = acc
    return out


def _flat(z: Tensor) -> Tensor:
    return z.reshape(-1, z.shape[-1])


def _signed_magnitude_update(w: Tensor, delta: Tensor, w_max: Optional[float]) -> None:
    """Change |w| by delta without ever flipping the sign of w."""
    mag = (w.abs() + delta).clamp_(min=0.0)
    if w_max is not None:
        mag.clamp_(max=w_max)
    w.copy_(torch.sign(w) * mag)


# --------------------------------------------------------------------------- #
# Recording of a forward pass (input to the local learning rules)
# --------------------------------------------------------------------------- #
@dataclass
class Recording:
    x: Tensor                  # (T, B, n_in)  inputs
    s_prev: Tensor             # (T, B, N)     recurrent pre-synaptic spikes used at each step
    spikes: Tensor             # (T, B, N)     post-synaptic spikes
    y: Tensor                  # (T, B, n_out) readout
    v: Optional[Tensor] = None  # (T, B, N)    pre-reset membrane (needs record_membrane)

    def rate_hz(self, dt: float) -> Tensor:
        """Per-neuron mean firing rate in Hz."""
        return self.spikes.mean(dim=(0, 1)) * (1000.0 / dt)


# --------------------------------------------------------------------------- #
# The network
# --------------------------------------------------------------------------- #
class RSNN(nn.Module):
    """Sparse recurrent E/I spiking network with a leaky linear readout.

    Parameters
    ----------
    n_in, n_rec, n_out : sizes of input, recurrent population and readout.
    exc_fraction       : fraction of recurrent neurons that are excitatory.
    connectivity       : recurrent connection probability. A float, or a dict
                         with keys "EE", "EI", "IE", "II" (post type first).
    input_connectivity : probability of each input->recurrent connection.
    neuron, neuron_kwargs : model for the recurrent neurons ("lif", "if",
                         "alif", "izhikevich", "stochastic_lif", "poisson", a
                         SpikingNeuron subclass, a factory ``f(n, dt=...)``, or
                         a pre-built population of size n_rec / n_exc).
    inh_neuron, inh_neuron_kwargs : if given, inhibitory neurons use this model
                         (a separate population); otherwise one population is
                         used for everything.
    dt                 : time step in ms (must match any pre-built neurons).
    dale               : enforce Dale's law (E columns >= 0, I columns <= 0).
    inh_gain           : initial inhibitory/excitatory weight ratio.
    input_scale, rec_scale : initial weight scales. They are in the units the
                         neuron expects: per-step charge for IF/LIF/ALIF,
                         current for Izhikevich (use much larger values there).
    spectral_radius    : if set, rescale initial recurrent weights to it.
    w_max              : optional cap on recurrent weight magnitude.
    tau_syn            : optional synaptic current time constant (ms).
    tau_out            : readout leak time constant (ms); None/0 = no leak.
    readout_from       : "all" or "exc" neurons feed the readout.
    train_input / train_recurrent / train_readout : which weights learn.
    seed               : seed for reproducible construction.
    """

    def __init__(
        self,
        n_in: int,
        n_rec: int,
        n_out: int,
        *,
        exc_fraction: float = 0.8,
        connectivity: Union[float, Dict[str, float]] = 0.1,
        input_connectivity: float = 1.0,
        neuron: NeuronSpec = "lif",
        neuron_kwargs: Optional[dict] = None,
        inh_neuron: Optional[NeuronSpec] = None,
        inh_neuron_kwargs: Optional[dict] = None,
        dt: float = 1.0,
        dale: bool = True,
        inh_gain: float = 2.0,
        input_scale: float = 1.0,
        rec_scale: float = 1.0,
        spectral_radius: Optional[float] = None,
        w_max: Optional[float] = None,
        tau_syn: Optional[float] = None,
        tau_out: Optional[float] = 20.0,
        readout_from: str = "all",
        readout_scale: float = 1.0,
        train_input: bool = True,
        train_recurrent: bool = True,
        train_readout: bool = True,
        seed: Optional[int] = None,
    ):
        super().__init__()
        if not 0.0 < exc_fraction <= 1.0:
            raise ValueError("exc_fraction must be in (0, 1]")
        if readout_from not in ("all", "exc"):
            raise ValueError("readout_from must be 'all' or 'exc'")
        self.n_in, self.n_rec, self.n_out = int(n_in), int(n_rec), int(n_out)
        self.dt = float(dt)
        self.dale = bool(dale)
        self.w_max = w_max
        self.train_input, self.train_recurrent, self.train_readout = (
            train_input, train_recurrent, train_readout)
        N = self.n_rec
        self.n_exc = min(N, max(1, int(round(exc_fraction * N))))
        self.n_inh = N - self.n_exc
        if inh_neuron is not None and self.n_inh == 0:
            raise ValueError("inh_neuron given but exc_fraction leaves no inhibitory neurons")

        gen = torch.Generator()
        gen.manual_seed(seed if seed is not None else int(torch.randint(0, 2**31 - 1, (1,))))

        # ---- neuron populations ---------------------------------------- #
        if inh_neuron is None:
            pops = [self._make_pop(neuron, neuron_kwargs, N)]
            self.slices = [(0, N)]
        else:
            pops = [self._make_pop(neuron, neuron_kwargs, self.n_exc),
                    self._make_pop(inh_neuron, inh_neuron_kwargs, self.n_inh)]
            self.slices = [(0, self.n_exc), (self.n_exc, N)]
        self.pops = nn.ModuleList(pops)

        # ---- cell types ------------------------------------------------- #
        is_exc = torch.zeros(N, dtype=torch.bool)
        is_exc[: self.n_exc] = True
        self.register_buffer("is_exc", is_exc)
        self.register_buffer("sign", torch.where(is_exc, 1.0, -1.0))

        # ---- fixed sparsity masks --------------------------------------- #
        P = self._probability_matrix(connectivity)
        rec_mask = (torch.rand(N, N, generator=gen) < P).float()
        rec_mask.fill_diagonal_(0.0)
        self.register_buffer("rec_mask", rec_mask)
        in_mask = (torch.rand(N, self.n_in, generator=gen) < float(input_connectivity)).float()
        self.register_buffer("in_mask", in_mask)
        out_mask = is_exc.float() if readout_from == "exc" else torch.ones(N)
        self.register_buffer("out_mask", out_mask)

        # ---- weights ----------------------------------------------------- #
        k_in = rec_mask.sum(1, keepdim=True).clamp(min=1.0)       # in-degree per neuron
        mag = torch.randn(N, N, generator=gen).abs() * rec_scale / k_in.sqrt()
        mag[:, self.n_exc:] *= inh_gain
        mag = mag * rec_mask
        if spectral_radius is not None:
            radius = torch.linalg.eigvals(mag * self.sign).abs().max()
            if radius > 0:
                mag = mag * (spectral_radius / radius)
        w_rec = mag if self.dale else mag * self.sign
        self.w_rec = nn.Parameter(w_rec, requires_grad=train_recurrent)

        n_eff = max(1.0, self.n_in * float(input_connectivity))
        w_in = torch.randn(N, self.n_in, generator=gen) * input_scale / math.sqrt(n_eff) * in_mask
        self.w_in = nn.Parameter(w_in, requires_grad=train_input)

        n_ro = max(1.0, float(out_mask.sum()))
        w_out = torch.randn(self.n_out, N, generator=gen) * readout_scale / math.sqrt(n_ro)
        self.w_out = nn.Parameter(w_out, requires_grad=train_readout)
        self.b_out = nn.Parameter(torch.zeros(self.n_out), requires_grad=train_readout)

        # ---- dynamics constants ------------------------------------------ #
        self.syn_decay = math.exp(-self.dt / tau_syn) if tau_syn else None
        self.kappa = math.exp(-self.dt / tau_out) if tau_out else 0.0   # readout leak
        self._state: Optional[Dict[str, Tensor]] = None

    # ------------------------------------------------------------------ #
    # construction helpers
    # ------------------------------------------------------------------ #
    def _make_pop(self, spec: NeuronSpec, kwargs: Optional[dict], n: int) -> SpikingNeuron:
        kw = dict(kwargs or {})
        if isinstance(spec, SpikingNeuron):
            if spec.n != n:
                raise ValueError(f"pre-built neuron has n={spec.n}, expected {n}")
            if abs(spec.dt - self.dt) > 1e-12:
                raise ValueError(f"pre-built neuron dt={spec.dt} != network dt={self.dt}")
            return spec
        kw.setdefault("dt", self.dt)
        if isinstance(spec, str):
            return make_neuron(spec, n, **kw)
        if isinstance(spec, type) and issubclass(spec, SpikingNeuron):
            return spec(n, **kw)
        if callable(spec):
            pop = spec(n, **kw)
            if not isinstance(pop, SpikingNeuron):
                raise TypeError("neuron factory must return a SpikingNeuron")
            return pop
        raise TypeError(f"cannot build a neuron population from {spec!r}")

    def _probability_matrix(self, connectivity: Union[float, Dict[str, float]]) -> Tensor:
        if isinstance(connectivity, dict):
            try:
                p = {k: float(connectivity[k]) for k in ("EE", "EI", "IE", "II")}
            except KeyError as e:
                raise ValueError("connectivity dict needs keys EE, EI, IE, II") from e
        else:
            p = {k: float(connectivity) for k in ("EE", "EI", "IE", "II")}
        if not all(0.0 <= v <= 1.0 for v in p.values()):
            raise ValueError("connection probabilities must be in [0, 1]")
        N, e = self.n_rec, self.n_exc
        P = torch.empty(N, N)
        P[:e, :e], P[:e, e:], P[e:, :e], P[e:, e:] = p["EE"], p["EI"], p["IE"], p["II"]  # (post, pre)
        return P

    # ------------------------------------------------------------------ #
    # weights
    # ------------------------------------------------------------------ #
    def effective_weights(self) -> Tuple[Tensor, Tensor, Tensor]:
        """(W_in, W_rec, W_out), differentiable, with masks and Dale's law applied."""
        W_in = self.w_in * self.in_mask
        if self.dale:
            W_rec = torch.relu(self.w_rec) * self.rec_mask * self.sign
        else:
            W_rec = self.w_rec * self.rec_mask
        W_out = self.w_out * self.out_mask
        return W_in, W_rec, W_out

    def rec_magnitude(self) -> Tensor:
        """|W_rec| with the sparsity mask applied."""
        w = torch.relu(self.w_rec) if self.dale else self.w_rec.abs()
        return w * self.rec_mask

    @property
    def density(self) -> float:
        return float(self.rec_mask.sum() / (self.n_rec * (self.n_rec - 1)))

    # --- parameter updates used by the non-gradient learning rules -------- #
    @torch.no_grad()
    def project(self, w_max: Optional[float] = None) -> None:
        """Re-impose Dale's law / weight cap after a direct update."""
        cap = w_max if w_max is not None else self.w_max
        if self.dale:
            self.w_rec.clamp_(min=0.0, max=cap)
        elif cap is not None:
            self.w_rec.clamp_(-cap, cap)

    @torch.no_grad()
    def apply_rec_magnitude_delta(self, delta: Tensor, w_max: Optional[float] = None) -> None:
        """Strengthen (delta > 0) or weaken recurrent synapses irrespective of
        their sign. Masked entries are untouched; signs are preserved."""
        delta = delta * self.rec_mask
        if self.dale:
            self.w_rec.add_(delta)
            self.project(w_max)
        else:
            _signed_magnitude_update(self.w_rec, delta, w_max if w_max is not None else self.w_max)

    @torch.no_grad()
    def apply_rec_effective_delta(self, delta: Tensor) -> None:
        """Add ``delta`` to the *effective* recurrent weights (gradient-style)."""
        delta = delta * self.rec_mask
        self.w_rec.add_(delta * self.sign if self.dale else delta)
        self.project()

    @torch.no_grad()
    def apply_in_magnitude_delta(self, delta: Tensor, w_max: Optional[float] = None) -> None:
        _signed_magnitude_update(self.w_in, delta * self.in_mask, w_max)

    @torch.no_grad()
    def apply_in_effective_delta(self, delta: Tensor) -> None:
        self.w_in.add_(delta * self.in_mask)

    @torch.no_grad()
    def apply_out_delta(self, d_w: Tensor, d_b: Optional[Tensor] = None) -> None:
        self.w_out.add_(d_w * self.out_mask)
        if d_b is not None:
            self.b_out.add_(d_b)

    # ------------------------------------------------------------------ #
    # state
    # ------------------------------------------------------------------ #
    def reset_state(self) -> None:
        self._state = None
        for p in self.pops:
            p.reset_state()

    def detach_state(self) -> None:
        if self._state is not None:
            self._state = {k: v.detach() for k, v in self._state.items()}
        for p in self.pops:
            p.detach_state()

    def _get_state(self, x: Tensor) -> Dict[str, Tensor]:
        B = x.shape[1]
        st = self._state
        if (st is not None and st["s"].shape == (B, self.n_rec)
                and st["s"].dtype == x.dtype and st["s"].device == x.device):
            return st
        for p in self.pops:
            p.reset_state()
        st = {"s": x.new_zeros(B, self.n_rec), "y": x.new_zeros(B, self.n_out)}
        if self.syn_decay is not None:
            st["i"] = x.new_zeros(B, self.n_rec)
        self._state = st
        return st

    # ------------------------------------------------------------------ #
    # dynamics
    # ------------------------------------------------------------------ #
    def _step_pops(self, cur: Tensor) -> Tensor:
        if len(self.pops) == 1:
            return self.pops[0](cur)
        return torch.cat([p(cur[:, a:b]) for p, (a, b) in zip(self.pops, self.slices)], dim=1)

    def _membrane(self) -> Tensor:
        if len(self.pops) == 1:
            return self.pops[0].membrane
        return torch.cat([p.membrane for p in self.pops], dim=1)

    def pseudo_derivative(self, v_pre: Tensor) -> Tensor:
        """Per-neuron d(spike)/d(v) for (T, B, N) pre-reset membrane."""
        parts = [p.pseudo_derivative(v_pre[..., a:b]) for p, (a, b) in zip(self.pops, self.slices)]
        return parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)

    def forward(self, x: Tensor, record: bool = False, record_membrane: bool = False):
        """Run the network over ``x`` of shape (T, B, n_in); state carries over
        between calls until ``reset_state()``. Returns the readout (T, B, n_out),
        plus a ``Recording`` if ``record`` or ``record_membrane``."""
        if x.dim() != 3 or x.shape[-1] != self.n_in:
            raise ValueError(f"expected input (T, B, {self.n_in}), got {tuple(x.shape)}")
        T, B, _ = x.shape
        N = self.n_rec
        st = self._get_state(x)
        W_in, W_rec, W_out = self.effective_weights()
        W_rec_t = W_rec.t()

        drive = torch.matmul(x, W_in.t())               # input current for all T in one GEMM
        spikes = x.new_empty(T, B, N)
        v_hist = x.new_empty(T, B, N) if record_membrane else None
        s0 = s = st["s"]
        i_syn = st.get("i")
        if record_membrane:
            for p in self.pops:
                p.record_pre_reset = True
        try:
            for t in range(T):
                cur = drive[t] + torch.matmul(s, W_rec_t)
                if i_syn is not None:
                    i_syn = i_syn * self.syn_decay + cur
                    cur = i_syn
                s = self._step_pops(cur)
                spikes[t] = s
                if v_hist is not None:
                    v_hist[t] = self._membrane()
        finally:
            if record_membrane:
                for p in self.pops:
                    p.record_pre_reset = False

        ro = torch.matmul(spikes, W_out.t()) + self.b_out   # readout for all T in one GEMM
        if self.kappa == 0.0:
            y, y_last = ro, ro[-1]
        else:
            y = torch.empty_like(ro)
            y_last = st["y"]
            for t in range(T):
                y_last = y_last * self.kappa + ro[t]
                y[t] = y_last

        st["s"], st["y"] = s, y_last
        if i_syn is not None:
            st["i"] = i_syn

        if not (record or record_membrane):
            return y
        s_prev = torch.cat([s0.unsqueeze(0), spikes[:-1]], dim=0)
        return y, Recording(x=x, s_prev=s_prev, spikes=spikes, y=y, v=v_hist)

    def extra_repr(self) -> str:
        return (f"in={self.n_in}, rec={self.n_rec} (E={self.n_exc}, I={self.n_inh}), "
                f"out={self.n_out}, density={self.density:.3f}, dale={self.dale}")


# --------------------------------------------------------------------------- #
# Error signals for the supervised, local rules (dL/dy, per time step)
# --------------------------------------------------------------------------- #
def mse_error(y: Tensor, target: Tensor) -> Tensor:
    """dL/dy for L = 0.5 * sum (y - target)^2 (the learners divide by batch)."""
    return y - target


def softmax_error(y: Tensor, labels: Tensor) -> Tensor:
    """dL/dy for per-time-step cross-entropy. ``labels``: (B,) or (T, B) class ids."""
    T, B, K = y.shape
    if labels.dim() == 1:
        labels = labels.unsqueeze(0).expand(T, B)
    return torch.softmax(y, dim=-1) - F.one_hot(labels, K).to(y.dtype)


# --------------------------------------------------------------------------- #
# Learning rules
# --------------------------------------------------------------------------- #
class Learner(ABC):
    """Common base: a learning rule bound to one network."""

    def __init__(self, model: RSNN):
        self.model = model

    @abstractmethod
    def update(self, *args, **kwargs):
        """Apply one weight update."""


# ---- 1. surrogate gradients + backpropagation through time ----------------- #
class SurrogateBPTT(Learner):
    """Supervised training by BPTT; spikes use the neurons' surrogate gradients.

    ``loss_fn(y, target) -> scalar``. It may be an ``nn.Module`` with its own
    parameters (a *learnable objective*); those are optimised jointly. Optional
    firing-rate regularisation keeps the network out of silence / runaway
    synchrony, and ``tbptt`` truncates the backward pass to chunks of that many
    steps. Gradients are clipped to ``grad_clip``.
    """

    def __init__(
        self,
        model: RSNN,
        loss_fn: Optional[Callable[[Tensor, Tensor], Tensor]] = None,
        optimizer: Optional[torch.optim.Optimizer] = None,
        lr: float = 1e-3,
        tbptt: Optional[int] = None,
        grad_clip: Optional[float] = 1.0,
        rate_target_hz: Optional[float] = None,
        rate_weight: float = 0.0,
        input_noise: float = 0.0,
    ):
        super().__init__(model)
        self.loss_fn = loss_fn if loss_fn is not None else F.mse_loss
        params = [p for p in model.parameters() if p.requires_grad]
        if isinstance(self.loss_fn, nn.Module):
            params += list(self.loss_fn.parameters())
        self.params = params
        self.opt = optimizer if optimizer is not None else torch.optim.Adam(params, lr=lr)
        self.tbptt = tbptt
        self.grad_clip = grad_clip
        self.rate_target_hz = rate_target_hz
        self.rate_weight = rate_weight
        self.input_noise = input_noise

    def update(self, x: Tensor, target: Tensor, carry_state: bool = False) -> float:
        """One optimisation pass over a (T, B, ...) sequence. Returns mean loss."""
        m = self.model
        m.train()
        if isinstance(self.loss_fn, nn.Module):
            self.loss_fn.train()
        if not carry_state:
            m.reset_state()
        T = x.shape[0]
        K = self.tbptt or T
        use_rate = self.rate_weight > 0.0 and self.rate_target_hz is not None
        total, chunks = 0.0, 0
        for a in range(0, T, K):
            xc, tc = x[a:a + K], target[a:a + K]
            if self.input_noise > 0.0:
                xc = xc + self.input_noise * torch.randn_like(xc)
            out = m(xc, record=use_rate)
            y, rec = out if use_rate else (out, None)
            loss = self.loss_fn(y, tc)
            if use_rate:
                p_target = self.rate_target_hz * m.dt * 1e-3
                loss = loss + self.rate_weight * ((rec.spikes.mean(dim=(0, 1)) - p_target) ** 2).mean()
            self.opt.zero_grad(set_to_none=True)
            loss.backward()
            if self.grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(self.params, self.grad_clip)
            self.opt.step()
            m.project()
            m.detach_state()
            total += float(loss.detach())
            chunks += 1
        return total / max(chunks, 1)


# ---- 2. pair-based STDP ---------------------------------------------------- #
class STDP(Learner):
    """Unsupervised pair-based STDP with exponential traces.

    LTP: a post-synaptic spike potentiates synapses from recently active
    pre-synaptic neurons; LTD: a pre-synaptic spike depresses synapses from
    recently active post-synaptic neurons. Changes act on weight *magnitude*
    (so excitatory and inhibitory synapses are both strengthened by LTP) and are
    summed over the chunk, then applied at its end. ``mu`` > 0 gives soft bounds
    (LTP ~ (w_max - w)^mu, LTD ~ w^mu; needs ``w_max``).

    targets: "rec" and/or "in" (input weights; input values act as pre-spikes).
    pre_types: restrict plasticity to "all", "exc" or "inh" pre-synaptic cells.
    """

    def __init__(
        self,
        model: RSNN,
        a_plus: float = 0.01,
        a_minus: float = 0.0105,
        tau_plus: float = 20.0,
        tau_minus: float = 20.0,
        lr: float = 1.0,
        w_max: Optional[float] = None,
        mu: float = 0.0,
        targets: Sequence[str] = ("rec",),
        pre_types: str = "all",
    ):
        super().__init__(model)
        if mu > 0.0 and w_max is None:
            raise ValueError("soft bounds (mu > 0) need w_max")
        if pre_types not in ("all", "exc", "inh"):
            raise ValueError("pre_types must be 'all', 'exc' or 'inh'")
        self.a_plus, self.a_minus, self.lr = a_plus, a_minus, lr
        self.d_plus = math.exp(-model.dt / tau_plus)
        self.d_minus = math.exp(-model.dt / tau_minus)
        self.w_max, self.mu = w_max, mu
        self.targets, self.pre_types = tuple(targets), pre_types

    # -- shared by RSTDP ---------------------------------------------------- #
    def _pair_terms(self, pre: Tensor, post: Tensor, post_w: Optional[Tensor] = None
                    ) -> Tuple[Tensor, Tensor]:
        """Return (P, D), each (n_post, n_pre): P = sum post(t) * pre_trace(t),
        D = sum post_trace(t-1) * pre(t); averaged over the batch."""
        B = post.shape[1]
        pre_tr = _causal_filter(pre, self.d_plus)
        post_tr = _causal_filter(post, self.d_minus)
        post_before = torch.zeros_like(post_tr)
        post_before[1:] = post_tr[:-1]
        post_ltp, post_ltd = post, post_before
        if post_w is not None:
            post_ltp, post_ltd = post * post_w, post_before * post_w
        P = _flat(post_ltp).t() @ _flat(pre_tr) / B
        D = _flat(post_ltd).t() @ _flat(pre) / B
        return P, D

    def _delta(self, P: Tensor, D: Tensor, w_mag: Tensor) -> Tensor:
        if self.mu == 0.0:
            ltp, ltd = P, D
        else:
            ltp = P * (self.w_max - w_mag).clamp(min=0.0).pow(self.mu)
            ltd = D * w_mag.pow(self.mu)
        return self.lr * (self.a_plus * ltp - self.a_minus * ltd)

    def _pre_filter(self) -> Optional[Tensor]:
        m = self.model
        if self.pre_types == "all":
            return None
        sel = m.is_exc if self.pre_types == "exc" else ~m.is_exc
        return sel.to(m.rec_mask.dtype).unsqueeze(0)       # (1, N): broadcast over post rows

    @torch.no_grad()
    def update(self, rec: Recording) -> None:
        m = self.model
        if "rec" in self.targets and m.train_recurrent:
            P, D = self._pair_terms(rec.s_prev, rec.spikes)
            delta = self._delta(P, D, m.rec_magnitude())
            sel = self._pre_filter()
            m.apply_rec_magnitude_delta(delta if sel is None else delta * sel, self.w_max)
        if "in" in self.targets and m.train_input:
            P, D = self._pair_terms(rec.x, rec.spikes)
            m.apply_in_magnitude_delta(self._delta(P, D, (m.w_in * m.in_mask).abs()), self.w_max)


# ---- 3. reward-modulated STDP --------------------------------------------- #
class RSTDP(STDP):
    """R-STDP: STDP eligibility, decayed with ``tau_elig`` toward the end of the
    chunk, multiplied by (reward - running baseline). Reward is a scalar or a
    per-sample (B,) tensor delivered after the chunk; positive reward reinforces
    the pairings that occurred, negative reward reverses them."""

    def __init__(self, model: RSNN, tau_elig: float = 200.0, baseline_decay: float = 0.95, **kw):
        super().__init__(model, **kw)
        self.elig_decay = math.exp(-model.dt / tau_elig)
        self.baseline_decay = baseline_decay
        self.baseline = 0.0

    @torch.no_grad()
    def update(self, rec: Recording, reward: Union[float, Tensor]) -> float:
        m = self.model
        T, B, _ = rec.spikes.shape
        r = torch.as_tensor(reward, dtype=rec.spikes.dtype, device=rec.spikes.device)
        r_mean = float(r.mean())
        r_eff = (r - self.baseline).reshape(1, -1, 1) if r.dim() else (r - self.baseline)
        self.baseline = self.baseline_decay * self.baseline + (1.0 - self.baseline_decay) * r_mean
        t_idx = torch.arange(T - 1, -1, -1, device=rec.spikes.device, dtype=rec.spikes.dtype)
        w_t = (self.elig_decay ** t_idx).reshape(T, 1, 1) * r_eff          # decay toward chunk end
        if "rec" in self.targets and m.train_recurrent:
            P, D = self._pair_terms(rec.s_prev, rec.spikes, w_t)
            delta = self._delta(P, D, m.rec_magnitude())
            sel = self._pre_filter()
            m.apply_rec_magnitude_delta(delta if sel is None else delta * sel, self.w_max)
        if "in" in self.targets and m.train_input:
            P, D = self._pair_terms(rec.x, rec.spikes, w_t)
            m.apply_in_magnitude_delta(self._delta(P, D, (m.w_in * m.in_mask).abs()), self.w_max)
        return r_mean


# ---- 4. three-factor rule / e-prop ---------------------------------------- #
def _membrane_decay(model: RSNN) -> float:
    """Mean membrane decay of the recurrent neurons (for eligibility traces)."""
    for p in model.pops:
        beta = getattr(p, "_beta", None)
        if callable(beta):
            b = beta()
            return float(b.mean()) if isinstance(b, Tensor) else float(b)
    return math.exp(-model.dt / 20.0)


class ThreeFactor(Learner):
    """Eligibility-trace x post-synaptic pseudo-derivative x learning signal.

        dW_ij = -lr * sum_t  L_i(t) * psi_i(t) * zbar_j(t)

    * pre-synaptic factor  zbar_j: pre-synaptic spikes low-passed with the
      membrane decay (the local "eligibility trace");
    * post-synaptic factor psi_i: pseudo-derivative of the spike function;
    * third factor L_i: either a learning signal broadcast from the output error
      through feedback weights (``feedback``: "random" or "symmetric"; this is
      e-prop), or any modulator you pass in (neuromodulator, reward prediction
      error, ...) via ``update(rec, modulator=...)``.

    The readout is trained with its exact local gradient. For the leaky
    readout, the sum over t is evaluated with an equivalent anti-causal filter
    of L, which makes the whole update one matmul per matrix; it is
    mathematically identical to accumulating the online rule across the chunk
    and applying it at the end. It ignores recurrent spike-to-spike credit
    paths and, for adaptive neurons, the threshold-adaptation eligibility, so it
    approximates BPTT (that is the e-prop trade-off) and is exact only for the
    readout. Needs ``record_membrane=True`` recordings.
    """

    def __init__(
        self,
        model: RSNN,
        lr: float = 1e-3,
        lr_out: Optional[float] = None,
        feedback: Union[str, Tensor] = "random",
        targets: Sequence[str] = ("rec", "in"),
        train_readout: bool = True,
        tau_trace: Optional[float] = None,
        kappa: Optional[float] = None,
        seed: int = 0,
    ):
        super().__init__(model)
        self.lr = lr
        self.lr_out = lr if lr_out is None else lr_out
        self.targets = tuple(targets)
        self.train_readout = train_readout
        self.alpha = math.exp(-model.dt / tau_trace) if tau_trace else None
        self.kappa = kappa
        self.feedback = feedback
        self._seed = seed
        self._B: Optional[Tensor] = None

    def _feedback(self) -> Tensor:
        """(N, n_out) matrix broadcasting the output error to each neuron."""
        m = self.model
        if isinstance(self.feedback, Tensor):
            return self.feedback.to(m.w_out.device, m.w_out.dtype)
        if self.feedback == "symmetric":
            return (m.w_out * m.out_mask).detach().t()
        if self.feedback == "random":
            if self._B is None:
                g = torch.Generator().manual_seed(self._seed)       # same matrix on any device
                self._B = torch.randn(m.n_rec, m.n_out, generator=g) / math.sqrt(m.n_out)
            if self._B.device != m.w_out.device or self._B.dtype != m.w_out.dtype:
                self._B = self._B.to(m.w_out.device, m.w_out.dtype)  # move once, then reuse
            return self._B
        raise ValueError("feedback must be 'random', 'symmetric' or a (N, n_out) tensor")

    @staticmethod
    def _as_modulator(mod: Union[float, Tensor], rec: Recording) -> Tensor:
        T, B, _ = rec.spikes.shape
        mod = torch.as_tensor(mod, dtype=rec.spikes.dtype, device=rec.spikes.device)
        if mod.dim() == 0:
            return mod.reshape(1, 1, 1).expand(T, B, 1)
        if mod.dim() == 1:                       # per-sample
            return mod.reshape(1, B, 1).expand(T, B, 1)
        if mod.dim() == 2:                       # (T, B)
            return mod.unsqueeze(-1)
        return mod                               # (T, B, 1) or (T, B, N)

    @torch.no_grad()
    def gradients(self, rec: Recording, error: Optional[Tensor] = None,
                  modulator: Optional[Union[float, Tensor]] = None) -> Dict[str, Tensor]:
        """Loss-gradient estimates (effective-weight units, (post, pre))."""
        m = self.model
        if modulator is None and error is None:
            raise ValueError("pass an output `error` (dL/dy) or an explicit `modulator`")
        T, B, N = rec.spikes.shape
        out: Dict[str, Tensor] = {}
        if self.targets:      # skipped entirely for readout-only training
            if rec.v is None:
                raise ValueError("ThreeFactor needs a recording made with record_membrane=True")
            psi = m.pseudo_derivative(rec.v.detach())
            if modulator is not None:
                L = self._as_modulator(modulator, rec)
                kappa = self.kappa if self.kappa is not None else 0.0
            else:
                L = torch.matmul(error, self._feedback().t())      # (T, B, N)
                kappa = self.kappa if self.kappa is not None else m.kappa
            alpha = self.alpha if self.alpha is not None else _membrane_decay(m)
            g = _anticausal_filter(L, kappa) * psi                  # (T, B, N)
            if "rec" in self.targets:
                out["rec"] = _flat(g).t() @ _flat(_causal_filter(rec.s_prev, alpha)) / B
            if "in" in self.targets:
                out["in"] = _flat(g).t() @ _flat(_causal_filter(rec.x, alpha)) / B
        if error is not None and self.train_readout:
            z_out = _causal_filter(rec.spikes, m.kappa)
            out["out"] = _flat(error).t() @ _flat(z_out) / B
            ones = _causal_filter(torch.ones(T, 1, 1, dtype=error.dtype, device=error.device), m.kappa)
            out["bias"] = (error * ones).sum(dim=(0, 1)) / B
        return out

    @torch.no_grad()
    def update(self, rec: Recording, error: Optional[Tensor] = None,
               modulator: Optional[Union[float, Tensor]] = None) -> Dict[str, Tensor]:
        m = self.model
        g = self.gradients(rec, error, modulator)
        if "rec" in g and m.train_recurrent:
            m.apply_rec_effective_delta(-self.lr * g["rec"])
        if "in" in g and m.train_input:
            m.apply_in_effective_delta(-self.lr * g["in"])
        if "out" in g and m.train_readout:
            m.apply_out_delta(-self.lr_out * g["out"], -self.lr_out * g["bias"])
        return g


# ---- 5. stabilising / homeostatic rules ---------------------------------- #
class InhibitorySTDP(Learner):
    """Vogels et al. (2011) inhibitory plasticity on I->E synapses.

        dw_ij = lr * (pre_trace_j * post_i + post_trace_i * pre_j - alpha * pre_j)

    with alpha set by the target rate, so inhibition grows until each
    excitatory neuron fires near ``target_rate_hz``. In an always-active
    network this is what lets the background be suppressed while strong,
    specific inputs still get through.
    """

    def __init__(self, model: RSNN, lr: float = 1e-3, target_rate_hz: float = 5.0,
                 tau: float = 20.0, w_max: Optional[float] = None):
        super().__init__(model)
        self.lr, self.w_max = lr, w_max
        self.decay = math.exp(-model.dt / tau)
        rho0 = target_rate_hz * model.dt * 1e-3                      # spikes per step
        self.alpha = 2.0 * rho0 / (1.0 - self.decay)

    @torch.no_grad()
    def update(self, rec: Recording) -> None:
        m = self.model
        if not m.train_recurrent or m.n_inh == 0:
            return
        B = rec.spikes.shape[1]
        pre, post = rec.s_prev, rec.spikes
        P = _flat(post).t() @ _flat(_causal_filter(pre, self.decay))
        Q = _flat(_causal_filter(post, self.decay)).t() @ _flat(pre)
        S = _flat(pre).sum(dim=0)                                      # (N_pre,)
        delta = self.lr * (P + Q - self.alpha * S.unsqueeze(0)) / B
        block = m.is_exc.to(delta.dtype).unsqueeze(1) * (~m.is_exc).to(delta.dtype).unsqueeze(0)
        m.apply_rec_magnitude_delta(delta * block, self.w_max)


class HomeostaticScaling(Learner):
    """Multiplicative synaptic scaling: each neuron scales its excitatory (and
    input) synapses up when it fires below ``target_rate_hz`` and down when above."""

    def __init__(self, model: RSNN, target_rate_hz: float = 10.0, eta: float = 0.1,
                 targets: Sequence[str] = ("rec", "in"), max_step: float = 2.0):
        super().__init__(model)
        self.target = target_rate_hz * model.dt * 1e-3
        self.eta = eta
        self.targets = tuple(targets)
        self.max_step = max_step

    @torch.no_grad()
    def update(self, rec: Recording) -> Tensor:
        m = self.model
        rate = rec.spikes.mean(dim=(0, 1))                                # (N,)
        err = (self.target - rate) / max(self.target, 1e-12)
        scale = torch.exp(self.eta * err).clamp(1.0 / self.max_step, self.max_step)
        if "rec" in self.targets and m.train_recurrent:
            factor = torch.ones_like(m.w_rec)
            factor[:, m.is_exc] = scale.unsqueeze(1)
            m.w_rec.mul_(factor)
            m.project()
        if "in" in self.targets and m.train_input:
            m.w_in.mul_(scale.unsqueeze(1))
        return rate


# --------------------------------------------------------------------------- #
# Self-test:  python rsnn.py
# --------------------------------------------------------------------------- #
def _selftest() -> None:
    torch.manual_seed(0)
    T, B, n_in, N, n_out = 40, 6, 12, 80, 3

    def spikes_in(p=0.2):
        return (torch.rand(T, B, n_in) < p).float()

    # 1. structure: size, E/I ratio, density, Dale's law, no self-connections
    net = RSNN(n_in, N, n_out, exc_fraction=0.75, connectivity=0.15, seed=1)
    assert (net.n_exc, net.n_inh) == (60, 20)
    assert abs(net.density - 0.15) < 0.04, net.density
    _, W, _ = net.effective_weights()
    assert (W[:, :60] >= 0).all() and (W[:, 60:] <= 0).all() and W.diagonal().abs().sum() == 0
    y = net(spikes_in())
    assert y.shape == (T, B, n_out) and torch.isfinite(y).all()

    # 2. per-type connectivity + spectral radius
    net2 = RSNN(n_in, N, n_out, connectivity=dict(EE=0.2, EI=0.4, IE=0.4, II=0.1),
                spectral_radius=0.9, seed=2)
    _, W2, _ = net2.effective_weights()
    assert abs(torch.linalg.eigvals(W2).abs().max().item() - 0.9) < 1e-3

    # 3. neuron models are interchangeable, incl. separate E / I models
    for kind, kw, scale in [("lif", {}, 1.0), ("if", {}, 1.0), ("alif", {}, 1.0),
                            ("stochastic_lif", {}, 1.0), ("izhikevich", {}, 10.0)]:
        n = RSNN(n_in, N, n_out, neuron=kind, neuron_kwargs=kw, input_scale=scale,
                 rec_scale=scale, seed=3)
        with torch.no_grad():
            yy = n(spikes_in())
        assert yy.shape == (T, B, n_out) and torch.isfinite(yy).all(), kind
    mixed = RSNN(n_in, N, n_out, neuron="izhikevich",
                 neuron_kwargs=dict(a=0.02, b=0.2, c=-65.0, d=8.0),
                 inh_neuron="izhikevich", inh_neuron_kwargs=dict(a=0.1, b=0.2, c=-65.0, d=2.0),
                 input_scale=10.0, rec_scale=10.0, seed=4)
    with torch.no_grad():
        assert mixed(spikes_in()).shape == (T, B, n_out)

    # 4. surrogate-gradient BPTT runs and gives finite gradients/loss
    net = RSNN(n_in, N, n_out, seed=5, input_scale=3.0, rec_scale=1.0)
    x, target = spikes_in(), torch.randn(T, B, n_out)
    bptt = SurrogateBPTT(net, lr=1e-2, tbptt=20, rate_target_hz=20.0, rate_weight=1.0)
    losses = [bptt.update(x, target) for _ in range(30)]
    assert all(math.isfinite(l) for l in losses)
    print(f"BPTT loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
    # ... with a learnable loss module
    class ScaledMSE(nn.Module):
        def __init__(self):
            super().__init__()
            self.log_w = nn.Parameter(torch.zeros(()))
        def forward(self, y, t):
            return torch.exp(self.log_w) * F.mse_loss(y, t)
    assert math.isfinite(SurrogateBPTT(net, loss_fn=ScaledMSE()).update(x, target))

    # 5. local rules: run, keep Dale's law and the sparsity mask intact
    net = RSNN(n_in, N, n_out, seed=6, input_scale=3.0, w_max=2.0)
    net.reset_state()
    with torch.no_grad():
        _, rec = net(x, record_membrane=True)
    before = net.w_rec.detach().clone()
    err = mse_error(rec.y, target)
    rules = [
        ("STDP", lambda: STDP(net, lr=0.5, w_max=2.0, mu=0.5, targets=("rec", "in")).update(rec)),
        ("R-STDP", lambda: RSTDP(net, lr=0.5, w_max=2.0).update(rec, reward=torch.rand(B))),
        ("3-factor", lambda: ThreeFactor(net, lr=1e-2).update(rec, error=err)),
        ("3-factor modulator", lambda: ThreeFactor(net, lr=1e-2).update(rec, modulator=torch.rand(B))),
        ("iSTDP", lambda: InhibitorySTDP(net, lr=1e-2).update(rec)),
        ("homeostasis", lambda: HomeostaticScaling(net).update(rec)),
    ]
    for name, run in rules:
        run()
        assert (net.w_rec >= 0).all(), name                           # Dale
        assert (net.w_rec * (1 - net.rec_mask)).abs().sum() == 0, name  # mask intact
        assert (net.w_rec <= 2.0 + 1e-6).all(), name                  # w_max
    assert not torch.equal(before, net.w_rec), "local rules changed nothing"

    # 6. e-prop readout gradient equals the autograd gradient exactly
    net = RSNN(n_in, N, n_out, seed=7, input_scale=3.0)
    net.reset_state()
    y, rec = net(x, record_membrane=True)
    loss = 0.5 * ((y - target) ** 2).sum() / B
    g_w, g_b = torch.autograd.grad(loss, [net.w_out, net.b_out], retain_graph=True)
    tf = ThreeFactor(net)
    g = tf.gradients(rec, error=(y - target).detach())
    assert torch.allclose(g["out"], g_w, rtol=1e-3, atol=1e-5), "readout gradient mismatch"
    assert torch.allclose(g["bias"], g_b, rtol=1e-3, atol=1e-5), "bias gradient mismatch"
    # recurrent: compare e-prop (symmetric feedback) with the BPTT gradient
    tf_sym = ThreeFactor(net, feedback="symmetric")
    g_rec = tf_sym.gradients(rec, error=(y - target).detach())["rec"]
    (g_bptt,) = torch.autograd.grad(loss, [net.w_rec])
    g_bptt_eff = g_bptt * net.sign * net.rec_mask        # raw -> effective-weight gradient
    cos = F.cosine_similarity(g_rec.flatten(), g_bptt_eff.flatten(), dim=0)
    print(f"e-prop vs BPTT recurrent gradient cosine similarity: {cos:.3f}")

    print("all rsnn self-tests passed")


if __name__ == "__main__":
    _selftest()

### Optional self-tests

In [ ]:
# Optional sanity checks of the neuron models and the network/learning rules (CPU, ~1 minute).
# Includes an exact check that the e-prop readout gradient equals the autograd gradient.
!python neurons.py
!python rsnn.py

## Benchmark protocol
```text
Benchmark sweeps for the RSNN: one figure per axis of variation, every figure
showing task accuracy.

    python tests.py                 # full run  (CPU: roughly 1-2 hours)
    python tests.py --quick         # tiny smoke run (a few minutes)
    python tests.py --only size ei  # subset of: size ei neuron method sparsity
    python tests.py --plots-only    # re-plot from saved results.json
    python tests.py --retune        # ignore cached hyper-parameters

Task (applied, time-series, noise-rejecting)
--------------------------------------------
Event detection in a noisy always-on input stream. 20 input channels emit
Poisson background spikes. In 2 of 3 trials a short burst of elevated rate
appears in channel group A (class 0) or group B (class 1) at a random time
(steps 5-22 of 60); in the third (class 2) there is only noise. The
network must report the class *after a delay*, so it has to ignore noise,
respond to the specific perturbation, and remember it. The objective is
cross-entropy of the readout averaged over the final window; the metric is
accuracy on a fixed held-out test set (chance = 33%).

Protocol
--------
* Every learning method trains the same network on the same task stream.
    reservoir  recurrent weights frozen, readout trained by the delta rule
    stdp       unsupervised STDP on recurrent weights + delta-rule readout
    rstdp      reward-modulated STDP (reward = +1 correct / -1 wrong) + delta-rule readout
    eprop      three-factor rule (e-prop) on input/recurrent/readout weights
    bptt       surrogate-gradient backpropagation through time, all weights
* Hyper-parameters (learning rates) are tuned BEFORE the sweeps by random
  search on a validation set at a reference configuration, separately for each
  learning method (and, for the neuron-type sweep, for each neuron model).
  Tuning seeds and the validation set are disjoint from the sweep seeds and the
  test set. Tuned values are cached in hparams.json.
* Network gain (input and recurrent weight scale) is calibrated per
  configuration so the untrained network fires near TARGET_HZ. Without this,
  changing size, density or E/I ratio would mostly change how active the
  network is rather than how well it computes.
* Each sweep point is repeated over several seeds (network + data stream).

Outputs (in --out): results.json, hparams.json, one PNG and one CSV per sweep.
The CSV is the table view of each figure.
```

In [ ]:
import argparse
import json
import math
import os
import random
import statistics
import sys
import time
import traceback
from dataclasses import asdict, dataclass, replace
from typing import Dict, List, Optional

import torch
import torch.nn.functional as F

from rsnn import RSNN, RSTDP, STDP, SurrogateBPTT, ThreeFactor

# Runs on the GPU when one is available. Override with --device, or by
# reassigning DEVICE (e.g. torch.device("cpu")) before calling run()/run_sweep().
# Data are generated on the CPU with seeded generators (so a seed means the same
# trials on any device) and moved over; everything else lives on DEVICE.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## The task
Burst detection in noise, plus the ideal-observer ceiling.

In [ ]:
# --------------------------------------------------------------------------- #
# Task
# --------------------------------------------------------------------------- #
N_IN, T_STEPS, WINDOW, N_CLASSES = 20, 60, 10, 3
# Calibrated with a NumPy replica of this network: an ideal observer reaches
# 100%, a frozen random reservoir with a trained readout ~60%, which leaves
# room for methods that also learn the recurrent weights. Weaker bursts
# (e.g. 0.10 / 0.22 / 8 steps) leave NO usable information in a frozen
# reservoir, so every method that cannot build memory sits at chance.
P_BG, P_SIG, BURST_LEN = 0.05, 0.40, 10
ONSET_LO, ONSET_HI = 5, 22
TARGET_HZ = 15.0
TASK_DESC = "3-class burst detection in noise, readout after a delay"


def make_batch(batch: int, gen: torch.Generator, device: Optional[torch.device] = None):
    """x: (T, B, N_IN) spikes; labels: (B,) in {0: burst A, 1: burst B, 2: noise only}.
    Generated on the CPU from ``gen`` and moved to ``device`` if given."""
    labels = torch.randint(0, N_CLASSES, (batch,), generator=gen)
    onset = torch.randint(ONSET_LO, ONSET_HI + 1, (batch,), generator=gen)
    t = torch.arange(T_STEPS).view(T_STEPS, 1, 1)
    active = ((t >= onset.view(1, batch, 1)) & (t < (onset + BURST_LEN).view(1, batch, 1))).float()
    half = N_IN // 2
    group = torch.zeros(batch, N_IN)
    group[labels == 0, :half] = 1.0
    group[labels == 1, half:] = 1.0
    p = P_BG + (P_SIG - P_BG) * active * group.unsqueeze(0)
    x = (torch.rand(T_STEPS, batch, N_IN, generator=gen) < p).float()
    if device is not None:
        x, labels = x.to(device), labels.to(device)
    return x, labels


def window_logits(y: torch.Tensor) -> torch.Tensor:
    return y[-WINDOW:].mean(dim=0)


def ce_loss(y: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Cross-entropy of the window-averaged readout; target is (T, B) labels."""
    return F.cross_entropy(window_logits(y), target[0].long())


def window_error(y: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """dL/dy_t of ``ce_loss``: nonzero only inside the final window."""
    p = torch.softmax(window_logits(y), dim=-1) - F.one_hot(labels, N_CLASSES).to(y.dtype)
    err = torch.zeros_like(y)
    err[-WINDOW:] = p.unsqueeze(0) / WINDOW
    return err


def oracle_accuracy(n_train: int = 2000, n_test: int = 256) -> float:
    """Ceiling: an ideal observer that reads the raw input spikes. Scan statistic =
    the largest sliding-window (BURST_LEN) spike count in each channel group;
    'noise only' if both are below a threshold fitted on training trials."""
    half = N_IN // 2

    def stats(x):
        c = torch.cat([torch.zeros(1, *x.shape[1:]), x.cumsum(0)])
        w = c[BURST_LEN:] - c[:-BURST_LEN]                          # (T-L+1, B, N_IN)
        return torch.stack([w[:, :, :half].sum(2).amax(0), w[:, :, half:].sum(2).amax(0)], dim=1)

    def predict(s, thr):
        top = s.argmax(dim=1)
        return torch.where(s.amax(dim=1) < thr, torch.full_like(top, N_CLASSES - 1), top)

    x, y = make_batch(n_train, torch.Generator().manual_seed(3))
    xt, yt = make_batch(n_test, torch.Generator().manual_seed(999))     # same set as the test split
    s, st = stats(x), stats(xt)
    thr = max(range(int(s.max()) + 2), key=lambda t: float((predict(s, t) == y).float().mean()))
    return float((predict(st, thr) == yt).float().mean())

## Configuration
Budgets (`QUICK` / `FULL`), search spaces and the sweeps.

In [ ]:
# --------------------------------------------------------------------------- #
# Configuration and budgets
# --------------------------------------------------------------------------- #
REF_NEURON = "lif"
METHODS = ("reservoir", "stdp", "rstdp", "eprop", "bptt")
MAIN = ("reservoir", "eprop", "bptt")        # series shown in the numeric-axis sweeps
NEURONS = ("if", "lif", "alif", "stochastic_lif", "izhikevich")


@dataclass(frozen=True)
class Cfg:
    n_rec: int = 100
    exc_fraction: float = 0.8
    connectivity: float = 0.1
    neuron: str = REF_NEURON
    method: str = "bptt"


@dataclass
class Budget:
    quick: bool
    ref_n: int            # network size used for tuning and for non-size sweeps
    iters: int            # training iterations per run
    batch: int
    n_eval: int           # test / validation set size
    seeds: int
    tune_trials: int      # random-search trials per method (reference neuron)
    tune_iters: int
    tune_seeds: int
    neuron_tune_trials: int  # trials per (method, neuron) in the neuron sweep
    rate_weight: float


FULL = Budget(False, ref_n=100, iters=120, batch=32, n_eval=256, seeds=3,
              tune_trials=8, tune_iters=120, tune_seeds=2, neuron_tune_trials=4, rate_weight=50.0)
QUICK = Budget(True, ref_n=40, iters=8, batch=16, n_eval=64, seeds=1,
               tune_trials=2, tune_iters=8, tune_seeds=1, neuron_tune_trials=1, rate_weight=50.0)

# log-uniform search ranges (the first trial is always the geometric centre)
# Readout step sizes must be large: the delta-rule gradient is small (the error is
# divided by WINDOW and the filtered spike features are ~0.3), so at lr_out <= 0.1
# the readout barely leaves its initialisation in ~100 iterations.
SPACES: Dict[str, Dict[str, tuple]] = {
    "reservoir": {"lr_out": (1e-2, 1e2)},
    "stdp": {"lr": (1e-3, 10.0), "lr_out": (1e-2, 1e2)},
    "rstdp": {"lr": (1e-3, 10.0), "lr_out": (1e-2, 1e2)},
    "eprop": {"lr": (1e-4, 1.0), "lr_out": (1e-2, 1e2)},
    "bptt": {"lr": (1e-4, 3e-2)},
}

SWEEPS = {
    "size": dict(field="n_rec", values=[25, 50, 100, 200, 400], quick=[20, 40], kind="lines",
                 xscale="log", xlabel="Recurrent network size (neurons)",
                 title="Accuracy vs network size", file="fig_network_size.png"),
    "ei": dict(field="exc_fraction", values=[0.5, 0.65, 0.8, 0.9, 1.0], quick=[0.6, 0.9], kind="lines",
               xscale="linear", xlabel="Excitatory fraction  E / (E + I)",
               title="Accuracy vs excitatory / inhibitory ratio", file="fig_ei_ratio.png"),
    "neuron": dict(field="neuron", values=list(NEURONS), quick=["lif", "izhikevich"], kind="bars",
                   xlabel="Neuron model", title="Accuracy vs neuron type", file="fig_neuron_type.png"),
    "method": dict(field="method", values=list(METHODS), quick=list(METHODS), kind="method",
                   xlabel="Learning method", title="Accuracy vs learning method",
                   file="fig_learning_method.png"),
    "sparsity": dict(field="connectivity", values=[0.02, 0.05, 0.1, 0.25, 0.5, 1.0], quick=[0.05, 0.5],
                     kind="lines", xscale="log", xlabel="Recurrent connection density",
                     title="Accuracy vs sparsity", file="fig_sparsity.png"),
}

METHOD_NAME = {"reservoir": "Reservoir (readout only)", "stdp": "STDP + readout",
               "rstdp": "R-STDP + readout", "eprop": "e-prop (three-factor)",
               "bptt": "Surrogate-gradient BPTT"}
METHOD_SHORT = {"reservoir": "Reservoir", "stdp": "STDP", "rstdp": "R-STDP",
                "eprop": "e-prop", "bptt": "BPTT"}
NEURON_NAME = {"if": "IF", "lif": "LIF", "alif": "Adaptive LIF",
               "stochastic_lif": "Stochastic LIF", "izhikevich": "Izhikevich"}

FAILURES: List[str] = []

# Fingerprint of everything that changes what a result means. Cached results and
# tuned hyper-parameters from a different task/search space are discarded.
TASK_SIG = json.loads(json.dumps(dict(
    n_in=N_IN, T=T_STEPS, window=WINDOW, p_bg=P_BG, p_sig=P_SIG, burst_len=BURST_LEN,
    onset=[ONSET_LO, ONSET_HI], target_hz=TARGET_HZ, spaces=SPACES)))

## One training run
Builds a network, calibrates its gain, trains with the chosen method and returns held-out accuracy. Everything runs on `DEVICE`.

In [ ]:
# --------------------------------------------------------------------------- #
# One training run
# --------------------------------------------------------------------------- #
def build_net(cfg: Cfg, gain: float, seed: int) -> RSNN:
    net = RSNN(N_IN, cfg.n_rec, N_CLASSES, exc_fraction=cfg.exc_fraction,
               connectivity=cfg.connectivity, neuron=cfg.neuron,
               input_scale=gain, rec_scale=gain, seed=seed)
    return net.to(DEVICE)       # before any optimiser is created


_GAIN_CACHE: Dict[tuple, float] = {}


@torch.no_grad()
def calibrate_gain(cfg: Cfg, seed: int, iters: int = 10) -> float:
    """Bisection (in log space) on the weight gain so the untrained network
    fires near TARGET_HZ on the task input."""
    key = (cfg.neuron, cfg.n_rec, cfg.exc_fraction, cfg.connectivity, seed)
    if key in _GAIN_CACHE:
        return _GAIN_CACHE[key]
    gen = torch.Generator().manual_seed(10_000 + seed)
    x, _ = make_batch(32, gen, DEVICE)
    lo, hi = math.log(1e-2), math.log(1e3)
    best_err, best_gain = float("inf"), math.exp((lo + hi) / 2)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        net = build_net(cfg, math.exp(mid), seed)
        net.reset_state()
        _, rec = net(x, record=True)
        rate = float(rec.rate_hz(net.dt).mean())
        err = abs(math.log((rate + 1e-3) / TARGET_HZ))
        if err < best_err:
            best_err, best_gain = err, math.exp(mid)
        if rate > TARGET_HZ:
            hi = mid
        else:
            lo = mid
    _GAIN_CACHE[key] = best_gain
    return best_gain


def _sync() -> None:
    """CUDA kernels are asynchronous; wait so wall-clock timings are honest."""
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def run(cfg: Cfg, hp: dict, bud: Budget, seed: int, split: str = "test",
        iters: Optional[int] = None) -> dict:
    """Train ``cfg.method`` and return accuracy on a fixed held-out set."""
    torch.manual_seed(seed)
    gain = calibrate_gain(cfg, seed)
    net = build_net(cfg, gain, seed)
    net.w_max = 3.0 * float(net.rec_magnitude().max())      # cap for the STDP family
    gen = torch.Generator().manual_seed(seed * 7919 + 1)
    x_eval, y_eval = make_batch(bud.n_eval, torch.Generator().manual_seed(555 if split == "val" else 999),
                                DEVICE)
    iters = iters or bud.iters
    m = cfg.method

    bptt = stdp = rstdp = eprop = readout = None
    if m == "bptt":
        bptt = SurrogateBPTT(net, loss_fn=ce_loss, lr=hp["lr"], grad_clip=1.0,
                             rate_target_hz=TARGET_HZ, rate_weight=bud.rate_weight)
    elif m == "eprop":
        eprop = ThreeFactor(net, lr=hp["lr"], lr_out=hp["lr_out"], targets=("rec", "in"))
    else:
        readout = ThreeFactor(net, lr=0.0, lr_out=hp["lr_out"], targets=())   # delta rule
        if m == "stdp":
            stdp = STDP(net, lr=hp["lr"], w_max=net.w_max, mu=0.5)
        elif m == "rstdp":
            rstdp = RSTDP(net, tau_elig=100.0, lr=hp["lr"], w_max=net.w_max, mu=0.5)

    _sync()
    t0 = time.time()
    for _ in range(iters):
        x, labels = make_batch(bud.batch, gen, DEVICE)
        if bptt is not None:
            bptt.update(x, labels.unsqueeze(0).expand(T_STEPS, -1))
            continue
        net.reset_state()
        with torch.no_grad():
            if eprop is not None:
                y, rec = net(x, record_membrane=True)
            else:
                y, rec = net(x, record=True)
        err = window_error(y, labels)
        if eprop is not None:
            eprop.update(rec, error=err)
        else:
            readout.update(rec, error=err)
            if stdp is not None:
                stdp.update(rec)
            elif rstdp is not None:
                correct = (window_logits(y).argmax(dim=1) == labels).float()
                rstdp.update(rec, reward=2.0 * correct - 1.0)
    _sync()
    seconds = time.time() - t0

    net.reset_state()
    with torch.no_grad():
        y, rec = net(x_eval, record=True)
    logits = torch.nan_to_num(window_logits(y))
    acc = float((logits.argmax(dim=1) == y_eval).float().mean())
    return dict(acc=acc, seconds=seconds, rate_hz=float(rec.rate_hz(net.dt).mean()), gain=gain)


def safe_run(cfg: Cfg, hp: dict, bud: Budget, seed: int, **kw) -> dict:
    """A failed run is recorded as NaN (and reported at the end), so one bad
    configuration cannot throw away an hours-long sweep."""
    try:
        return run(cfg, hp, bud, seed, **kw)
    except Exception:
        traceback.print_exc()
        FAILURES.append(f"{cfg} seed={seed}")
        return dict(acc=float("nan"), seconds=float("nan"), rate_hz=float("nan"), gain=float("nan"))

## Hyper-parameter search
Random search for learning rates on a validation set, done before the sweeps and cached.

In [ ]:
# --------------------------------------------------------------------------- #
# Hyper-parameter search (done before the sweeps, cached on disk)
# --------------------------------------------------------------------------- #
def sample_hp(space: Dict[str, tuple], n: int, rng: random.Random) -> List[dict]:
    trials = [{k: math.sqrt(lo * hi) for k, (lo, hi) in space.items()}]
    while len(trials) < n:
        trials.append({k: math.exp(rng.uniform(math.log(lo), math.log(hi)))
                       for k, (lo, hi) in space.items()})
    return trials


class HPStore:
    """Tuned hyper-parameters keyed by 'method|neuron', persisted as JSON."""

    def __init__(self, path: str, retune: bool = False):
        self.path = path
        self.data: Dict[str, dict] = {}
        if os.path.exists(path) and not retune:
            with open(path) as f:
                self.data = json.load(f)
            if self.data.get("_task") != TASK_SIG:
                print(f"{path}: tuned for a different task / search space, discarding", flush=True)
                self.data = {}

    def get(self, method: str, neuron: str, bud: Budget) -> dict:
        key = f"{method}|{neuron}"
        if key not in self.data:
            self.data[key] = self._tune(method, neuron, bud)
            self.data["_task"] = TASK_SIG
            with open(self.path, "w") as f:
                json.dump(self.data, f, indent=2)
        return self.data[key]["hp"]

    @staticmethod
    def _tune(method: str, neuron: str, bud: Budget) -> dict:
        cfg = Cfg(n_rec=bud.ref_n, neuron=neuron, method=method)
        n_trials = bud.tune_trials if neuron == REF_NEURON else bud.neuron_tune_trials
        rng = random.Random(f"{method}|{neuron}")
        trials = []
        for i, hp in enumerate(sample_hp(SPACES[method], n_trials, rng)):
            accs = [safe_run(cfg, hp, bud, seed=1000 + s, split="val", iters=bud.tune_iters)["acc"]
                    for s in range(bud.tune_seeds)]
            accs = [a for a in accs if not math.isnan(a)]
            val = statistics.mean(accs) if accs else float("nan")
            trials.append(dict(hp=hp, val_acc=val))
            print(f"[tune {method}|{neuron}] trial {i + 1}/{n_trials} "
                  f"{ {k: round(v, 5) for k, v in hp.items()} } val acc {val:.3f}", flush=True)
        best = max(trials, key=lambda t: -1.0 if math.isnan(t["val_acc"]) else t["val_acc"])
        return dict(hp=best["hp"], val_acc=best["val_acc"], trials=trials)

## Sweep machinery

In [ ]:
# --------------------------------------------------------------------------- #
# Sweeps
# --------------------------------------------------------------------------- #
def run_sweep(name: str, bud: Budget, store: HPStore) -> List[dict]:
    spec = SWEEPS[name]
    values = spec["quick"] if bud.quick else spec["values"]
    base = Cfg(n_rec=bud.ref_n)
    records: List[dict] = []
    for x in values:
        methods = (x,) if name == "method" else MAIN
        for m in methods:
            neuron = x if name == "neuron" else REF_NEURON
            cfg = replace(base, method=m, **({} if name == "method" else {spec["field"]: x}))
            hp = store.get(m, neuron, bud)
            for s in range(bud.seeds):
                r = safe_run(cfg, hp, bud, seed=s)
                records.append(dict(x=x, method=m, seed=s, **r))
                print(f"[{name}] x={x} method={m} seed={s}: acc={r['acc']:.3f} "
                      f"rate={r['rate_hz']:.1f}Hz  {r['seconds']:.1f}s", flush=True)
    return records


def aggregate(records: List[dict]) -> Dict[str, Dict[object, dict]]:
    """{method: {x: {acc, std, n, seconds, rate_hz}}}  (NaN runs are ignored)."""
    groups: Dict[tuple, List[dict]] = {}
    for r in records:
        groups.setdefault((r["method"], r["x"]), []).append(r)
    out: Dict[str, Dict[object, dict]] = {}
    for (m, x), rs in groups.items():
        ok = [r for r in rs if not math.isnan(r["acc"])]
        acc = [r["acc"] for r in ok]
        out.setdefault(m, {})[x] = dict(
            acc=statistics.mean(acc) if acc else float("nan"),
            std=statistics.stdev(acc) if len(acc) > 1 else 0.0,
            n=len(acc),
            seconds=statistics.mean([r["seconds"] for r in ok]) if ok else float("nan"),
            rate_hz=statistics.mean([r["rate_hz"] for r in ok]) if ok else float("nan"),
        )
    return out


TABLE_COLUMNS = ("sweep", "x", "method", "acc_mean", "acc_std", "n_seeds", "train_seconds", "rate_hz")


def table_rows(sweep: str, agg: Dict[str, Dict[object, dict]]) -> List[dict]:
    """The table view of a figure (also what ends up in the CSV)."""
    return [dict(sweep=sweep, x=x, method=m, acc_mean=round(a["acc"], 4), acc_std=round(a["std"], 4),
                 n_seeds=a["n"], train_seconds=round(a["seconds"], 2), rate_hz=round(a["rate_hz"], 2))
            for m, by_x in agg.items() for x, a in by_x.items()]


def write_csv(path: str, sweep: str, agg: Dict[str, Dict[object, dict]]) -> None:
    with open(path, "w") as f:
        f.write(",".join(TABLE_COLUMNS) + "\n")
        for row in table_rows(sweep, agg):
            f.write(",".join(str(row[c]) for c in TABLE_COLUMNS) + "\n")

## Plotting
Light-mode validated palette, legend + direct labels + table view.

In [ ]:
# --------------------------------------------------------------------------- #
# Plotting (matplotlib, imported lazily so the experiments run without it)
# --------------------------------------------------------------------------- #
# Light-mode tokens and categorical slots 1-3 of the validated reference
# palette (all-pairs CVD dE >= 9.2, normal-vision dE >= 24). Colour follows the
# method, in every figure. The aqua slot is under 3:1 contrast on the surface,
# so every figure also carries a legend, direct labels where they fit, distinct
# marker shapes, and a CSV table view.
SURFACE, PRIMARY, SECONDARY, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE = "#e1e0d9", "#c3c2b7"
COLOR = {"reservoir": "#2a78d6", "eprop": "#eb6834", "bptt": "#1baf7a"}
MARKER = {"reservoir": "o", "eprop": "s", "bptt": "^"}
SINGLE = "#2a78d6"        # one-series bar charts use categorical slot 1
CHANCE = 1.0 / N_CLASSES
DPI = 200
SHOW = False              # True in a notebook: draw figures inline instead of only saving


def _mpl():
    import matplotlib
    if not SHOW:
        matplotlib.use("Agg")      # headless; in a notebook keep the inline backend
    import matplotlib.pyplot as plt
    return plt


def _finish(fig, plt, path):
    """Save to ``path`` (if given) and/or display inline (if SHOW)."""
    if path:
        fig.savefig(path, dpi=DPI, facecolor=SURFACE)
    if SHOW:
        plt.show()
    plt.close(fig)


def _new_axes(fig, rect, ylim, ylabel, pct=True):
    from matplotlib.ticker import FuncFormatter
    ax = fig.add_axes(rect, facecolor=SURFACE)
    ax.set_ylim(*ylim)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(BASELINE)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8, linestyle="-")
    ax.set_axisbelow(True)
    ax.tick_params(axis="both", length=0, labelsize=8, labelcolor=SECONDARY, pad=5)
    if pct:
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_ylabel(ylabel, fontsize=8.5, color=SECONDARY, labelpad=6)
    return ax


def _titles(fig, title, subtitle):
    fig.text(0.02, 0.955, title, fontsize=12, fontweight="semibold", color=PRIMARY, ha="left", va="top")
    fig.text(0.02, 0.905, subtitle, fontsize=8, color=SECONDARY, ha="left", va="top")


def _fmt(v):
    return str(v) if isinstance(v, int) else f"{v:g}"


def _refs(ax, meta, x_left):
    """Reference lines: chance, and the ideal-observer ceiling when known."""
    ax.axhline(CHANCE, color=MUTED, linewidth=1.0, zorder=1)
    ax.text(x_left, CHANCE + 0.015, "chance", fontsize=7.5, color=MUTED, ha="left", va="bottom")
    ceiling = meta.get("oracle")
    if ceiling:
        ax.axhline(ceiling, color=MUTED, linewidth=1.0, zorder=1)
        ax.text(x_left, ceiling - 0.015, "ideal observer (reads the raw input)", fontsize=7.5,
                color=MUTED, ha="left", va="top")


def _subtitle(meta) -> str:
    return (f"{TASK_DESC}.  Mean ± s.d. over {meta['seeds']} seeds, test set n={meta['n_eval']}.  "
            f"Chance = {CHANCE:.0%}.")


def plot_lines(path, spec, agg, values, meta):
    import numpy as np
    from matplotlib.lines import Line2D
    plt = _mpl()
    fig = plt.figure(figsize=(6.6, 4.6), dpi=DPI, facecolor=SURFACE)
    ax = _new_axes(fig, [0.11, 0.27, 0.80, 0.56], (0.0, 1.0), "Test accuracy")
    xs = np.array(values, dtype=float)
    if spec["xscale"] == "log":
        ax.set_xscale("log")
    if spec["xscale"] == "log":
        ax.set_xlim(xs.min() / 1.15, xs.max() * 1.9)             # room on the right for end labels
    else:
        pad = xs.max() - xs.min()
        ax.set_xlim(xs.min() - 0.03 * pad, xs.max() + 0.22 * pad)
    ax.set_xticks(xs)
    ax.set_xticklabels([_fmt(v) for v in values])
    ax.minorticks_off()
    ax.set_xlabel(spec["xlabel"], fontsize=8.5, color=SECONDARY, labelpad=6)

    _refs(ax, meta, xs.min())

    ends = []
    for m in MAIN:
        by_x = agg.get(m, {})
        mean = np.array([by_x.get(v, {}).get("acc", np.nan) for v in values])
        std = np.array([by_x.get(v, {}).get("std", np.nan) for v in values])
        ax.fill_between(xs, mean - std, mean + std, color=COLOR[m], alpha=0.10, linewidth=0, zorder=2)
        ax.plot(xs, mean, color=COLOR[m], linewidth=1.8, solid_capstyle="round", solid_joinstyle="round",
                marker=MARKER[m], markersize=7, markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=3)
        valid = np.where(~np.isnan(mean))[0]
        if len(valid):
            ends.append((mean[valid[-1]], xs[valid[-1]], m))

    # direct labels at the line ends, only when they do not collide (else the legend carries identity)
    ys = sorted(e[0] for e in ends)
    if len(ends) > 1 and min(b - a for a, b in zip(ys, ys[1:])) >= 0.07:
        for y, x, m in ends:
            ax.annotate(METHOD_SHORT[m], (x, y), xytext=(8, 0), textcoords="offset points",
                        fontsize=8, color=PRIMARY, va="center", ha="left")

    handles = [Line2D([0], [0], color=COLOR[m], linewidth=1.8, marker=MARKER[m], markersize=7,
                      markeredgecolor=SURFACE, markeredgewidth=1.5, label=METHOD_NAME[m]) for m in MAIN]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=8,
               labelcolor=SECONDARY, bbox_to_anchor=(0.5, 0.01), handlelength=2.2, columnspacing=1.6)
    _titles(fig, spec["title"], _subtitle(meta))
    _finish(fig, plt, path)


def _round_bar(ax, x, h, w, color, r_px=None):
    """Bar with a rounded data-end (4 css px) and a square baseline."""
    r_px = 4.0 * DPI / 100.0 if r_px is None else r_px
    from matplotlib.patches import FancyBboxPatch, Rectangle
    if not (h > 0):
        return
    bb = ax.get_window_extent()
    (x0, x1), (y0, y1) = ax.get_xlim(), ax.get_ylim()
    ux, uy = (x1 - x0) / bb.width, (y1 - y0) / bb.height          # data units per device pixel
    rx, ry = r_px * ux, r_px * uy
    shrink = min(1.0, (h / 2) / ry, (w / 2) / rx)
    rx, ry = rx * shrink, ry * shrink
    ax.add_patch(FancyBboxPatch((x - w / 2, 0), w, h, boxstyle=f"round,pad=0,rounding_size={rx}",
                                mutation_aspect=ry / rx, facecolor=color, edgecolor="none", linewidth=0,
                                zorder=3))
    ax.add_patch(Rectangle((x - w / 2, 0), w, ry, facecolor=color, edgecolor="none", linewidth=0, zorder=3))


def _bar_panel(fig, rect, cats, cat_labels, series, ylim, ylabel, pct, label_tips, xlabel):
    """series: list of (name, color, means, stds). Bars <= 24 css px wide, 2 px gaps."""
    import numpy as np
    ax = _new_axes(fig, rect, ylim, ylabel, pct=pct)
    n, k = len(cats), len(series)
    ax.set_xlim(-0.6, n - 0.4)
    ax.set_xticks(range(n))
    ax.set_xticklabels(cat_labels)
    ax.set_xlabel(xlabel, fontsize=8.5, color=SECONDARY, labelpad=6)
    slot = min(0.24, 0.72 / k)
    bb = ax.get_window_extent()
    gap = (2.0 * DPI / 100.0) * (ax.get_xlim()[1] - ax.get_xlim()[0]) / bb.width   # 2 css px
    span = ylim[1] - ylim[0]
    for j, (name, color, means, stds) in enumerate(series):
        for i in range(n):
            m, s = means[i], stds[i]
            if m is None or np.isnan(m):
                continue
            xc = i + (j - (k - 1) / 2) * slot
            _round_bar(ax, xc, m, slot - gap, color)
            if s and not np.isnan(s) and s > 0:
                ax.plot([xc, xc], [max(m - s, 0), m + s], color=SECONDARY, linewidth=1.0,
                        solid_capstyle="butt", zorder=4)
            if label_tips:
                txt = f"{m:.0%}" if pct else f"{m:.0f}s"
                ax.text(xc, m + (s if s and not np.isnan(s) else 0) + 0.015 * span, txt,
                        fontsize=8, color=PRIMARY, ha="center", va="bottom", zorder=5)
    return ax


def plot_bars(path, spec, agg, values, meta):
    """Neuron-type figure: grouped bars, one series per learning method."""
    import numpy as np
    from matplotlib.patches import Patch
    plt = _mpl()
    fig = plt.figure(figsize=(6.6, 4.6), dpi=DPI, facecolor=SURFACE)
    series = []
    for m in MAIN:
        by_x = agg.get(m, {})
        series.append((METHOD_NAME[m], COLOR[m],
                       [by_x.get(v, {}).get("acc", np.nan) for v in values],
                       [by_x.get(v, {}).get("std", np.nan) for v in values]))
    ax = _bar_panel(fig, [0.11, 0.27, 0.86, 0.56], values, [NEURON_NAME.get(v, v) for v in values],
                    series, (0.0, 1.0), "Test accuracy", True, False, spec["xlabel"])
    _refs(ax, meta, ax.get_xlim()[0] + 0.02)
    handles = [Patch(facecolor=COLOR[m], edgecolor="none", label=METHOD_NAME[m]) for m in MAIN]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=8,
               labelcolor=SECONDARY, bbox_to_anchor=(0.5, 0.01), handlelength=1.2, columnspacing=1.6)
    _titles(fig, spec["title"], _subtitle(meta))
    _finish(fig, plt, path)


def plot_method(path, spec, agg, values, meta):
    """Learning-method figure: accuracy and training time as two panels (never a dual axis)."""
    import numpy as np
    plt = _mpl()
    fig = plt.figure(figsize=(8.4, 4.4), dpi=DPI, facecolor=SURFACE)
    names = [METHOD_SHORT[v] for v in values]
    acc = [agg.get(v, {}).get(v, {}).get("acc", np.nan) for v in values]
    std = [agg.get(v, {}).get(v, {}).get("std", np.nan) for v in values]
    sec = [agg.get(v, {}).get(v, {}).get("seconds", np.nan) for v in values]
    ax = _bar_panel(fig, [0.08, 0.17, 0.41, 0.63], values, names, [("Test accuracy", SINGLE, acc, std)],
                    (0.0, 1.0), "Test accuracy", True, True, spec["xlabel"])
    _refs(ax, meta, ax.get_xlim()[0] + 0.02)
    top = max([s for s in sec if not np.isnan(s)] or [1.0]) * 1.25
    _bar_panel(fig, [0.58, 0.17, 0.40, 0.63], values, names, [("Training time", SINGLE, sec, [0] * len(sec))],
               (0.0, top), "Training time (s)", False, True, spec["xlabel"])
    _titles(fig, spec["title"], _subtitle(meta) + "  Right: wall-clock training time.")
    _finish(fig, plt, path)


PLOTTERS = {"lines": plot_lines, "bars": plot_bars, "method": plot_method}


def make_outputs(out: Optional[str], name: str, records: List[dict], meta: dict) -> List[dict]:
    """Draw (and, if ``out`` is a directory, save) the figure for one sweep.
    With ``out=None`` nothing touches the disk: set SHOW=True to display it.
    Returns the table rows behind the figure."""
    spec = SWEEPS[name]
    values = spec["quick"] if meta["quick"] else spec["values"]
    agg = aggregate(records)
    png = os.path.join(out, spec["file"]) if out else None
    if out:
        write_csv(os.path.join(out, spec["file"].replace(".png", ".csv")), name, agg)
        print(f"wrote {spec['file']} (+ csv)", flush=True)
    PLOTTERS[spec["kind"]](png, spec, agg, values, meta)
    return table_rows(name, agg)


def preflight(bud: Budget, ceiling: float) -> None:
    """Cheap sanity check before the long run: can a *frozen* reservoir with a
    trained readout beat chance on this task? If not, every method that cannot
    build memory sits at chance and the comparison says nothing."""
    r = safe_run(Cfg(n_rec=bud.ref_n, method="reservoir"), {"lr_out": 1.0}, bud, seed=0, split="val")
    print(f"preflight: chance {CHANCE:.0%} | frozen reservoir {r['acc']:.0%} "
          f"at {r['rate_hz']:.1f} Hz | ideal observer {ceiling:.0%}", flush=True)
    if not r["acc"] > CHANCE + 0.08:
        print("WARNING: the frozen reservoir is at chance, so the input does not survive in "
              "its activity. Methods that cannot learn memory will show chance accuracy. "
              "Make the burst stronger (P_SIG), longer (BURST_LEN) or later (ONSET_*), or "
              "give the neurons slower dynamics.", file=sys.stderr, flush=True)
    if bud.quick:
        print("(quick budget: too few iterations for this number to be meaningful)")

## Setup
Picks the device and budget, computes the ideal-observer ceiling, and runs a quick preflight check (a frozen reservoir should beat chance).

In [ ]:
import json, os
from dataclasses import asdict, replace

import pandas as pd
from IPython.display import display

SHOW = True        # draw figures inline (no files are written for the figures)
DPI = 110          # screen resolution; the command-line default of 200 is for saved files

if DEVICE_OVERRIDE != "auto":
    DEVICE = torch.device(DEVICE_OVERRIDE)
else:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("running on:", DEVICE, f"({torch.cuda.get_device_name(DEVICE)})" if DEVICE.type == "cuda" else "")

BUD = QUICK if QUICK_RUN else FULL
if SEEDS_OVERRIDE:
    BUD = replace(BUD, seeds=SEEDS_OVERRIDE)

tag = "_quick" if QUICK_RUN else ""
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/biolearning_results{tag}"     # PNG + CSV copies are saved here too
else:
    OUT = None                                                   # display only
WORK = OUT or f"/content/biolearning_results{tag}"
os.makedirs(WORK, exist_ok=True)
RES_PATH = os.path.join(WORK, "results.json")

RESULTS = {"meta": {}, "sweeps": {}}
if os.path.exists(RES_PATH):
    RESULTS = json.load(open(RES_PATH))
    if RESULTS.get("meta", {}).get("task") != TASK_SIG:
        print("saved results belong to a different task definition: discarding them")
        RESULTS = {"meta": {}, "sweeps": {}}

ORACLE = oracle_accuracy(n_test=BUD.n_eval)
RESULTS["meta"] = dict(asdict(BUD), task=TASK_SIG, oracle=ORACLE, device=str(DEVICE))
STORE = HPStore(os.path.join(WORK, "hparams.json"))
FIG_ROWS = {}


def run_and_show(name, force=False):
    """Run one sweep (skipped if already done), draw its figure inline, show its table."""
    if force or name not in RESULTS["sweeps"]:
        RESULTS["sweeps"][name] = run_sweep(name, BUD, STORE)
        json.dump(RESULTS, open(RES_PATH, "w"), indent=1)       # resume point
    FIG_ROWS[name] = make_outputs(OUT, name, RESULTS["sweeps"][name], RESULTS["meta"])
    display(pd.DataFrame(FIG_ROWS[name]).drop(columns="sweep"))
    if FAILURES:
        print(f"{len(FAILURES)} run(s) failed so far and are recorded as NaN; see the tracebacks above")


preflight(BUD, ORACLE)

## Sweep 1: network size
Number of recurrent neurons, 25 to 400. Larger networks give more room for memory and benefit most from the GPU.

In [ ]:
run_and_show("size")

## Sweep 2: excitatory / inhibitory ratio
Fraction of neurons that are excitatory (50% to 100%). 100% means no inhibition.

In [ ]:
run_and_show("ei")

## Sweep 3: neuron type
IF, LIF, adaptive LIF, stochastic LIF and Izhikevich neurons; learning rates are tuned separately for each model.

In [ ]:
run_and_show("neuron")

## Sweep 4: learning method
Reservoir (readout only), STDP, R-STDP, e-prop and surrogate-gradient BPTT. Accuracy on the left, wall-clock training time on the right.

In [ ]:
run_and_show("method")

## Sweep 5: sparsity
Recurrent connection density from 2% to 100%.

In [ ]:
run_and_show("sparsity")

## Summary

In [ ]:
# Everything in one table, plus the tuned hyper-parameters.
all_rows = [r for rows in FIG_ROWS.values() for r in rows]
if all_rows:
    display(pd.DataFrame(all_rows))
print("tuned hyper-parameters:")
display(pd.DataFrame({k: v["hp"] for k, v in STORE.data.items() if k != "_task"}).T)
if FAILURES:
    print(f"\n{len(FAILURES)} run(s) failed (recorded as NaN):")
    for f in FAILURES[:20]:
        print("  ", f)